# Cálculo de variables objetivo - TIBAITATA

Este notebook preparará las variables necesarias para el cálculo de evapotranspiración de referencia mediante FAO-56.

## Importaciones

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Configuración de visualización

Se muestran todas las columnas de los DataFrames y se establece una precisión numérica uniforme para facilitar la inspección de resultados.

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
plt.style.use("seaborn-v0_8-whitegrid")

## Rutas de datos

Se declaran las rutas y se cargan los datasets sin aplicar limpieza ni preprocesamiento.

In [3]:
DATASET_LIMPIO_PATH = "./Datasets/TibaitataLimpio.pkl"
DATASET_ESTACION_PATH = "./Datasets/tibaitata.csv"

df_tibaitata_limpio = pd.read_pickle(DATASET_LIMPIO_PATH)
df_estacion = pd.read_csv(DATASET_ESTACION_PATH)

print("Dataset limpio:", df_tibaitata_limpio.shape)
print("Estacion:", df_estacion.shape)

Dataset limpio: (14545, 9)
Estacion: (1, 20)


## Constantes físicas para FAO-56

- `SOLAR_CONSTANT`: radiación extraterrestre solar, expresada en MJ m⁻² min⁻¹.
- `STEFAN_BOLTZMANN`: constante de Stefan-Boltzmann para la radiación neta de onda larga, en MJ K⁻⁴ m⁻² día⁻¹.
- `LATENT_HEAT_VAPORIZATION`: energía necesaria para vaporizar un kilogramo de agua, en MJ kg⁻¹.
- `SPECIFIC_HEAT_AIR`: calor específico del aire a presión constante, en MJ kg⁻¹ °C⁻¹.
- `EPSILON`: relación entre las masas molares del vapor de agua y el aire seco, adimensional.
- `KELVIN_OFFSET`: desplazamiento para convertir temperaturas de °C a K.
- `SECONDS_PER_DAY`: número de segundos de un día, útil para conversiones de tiempo.


In [4]:
SOLAR_CONSTANT = 0.0820  # MJ m^-2 min^-1
STEFAN_BOLTZMANN = 4.903e-9  # MJ K^-4 m^-2 day^-1
LATENT_HEAT_VAPORIZATION = 2.45  # MJ kg^-1
SPECIFIC_HEAT_AIR = 1.013e-3  # MJ kg^-1 °C^-1
EPSILON = 0.622  # adimensional
KELVIN_OFFSET = 273.15  # K por °C
SECONDS_PER_DAY = 86_400  # s day^-1

## Organización de variables meteorológicas

Se transforma el dataset limpio de formato largo a formato ancho. Antes del pivotado se revisan las claves duplicadas `Fecha`-`parametro_limpio`; si existen, el proceso se detiene para evitar promediar o eliminar registros automáticamente.

In [5]:
parametros_et0 = {
    "Humedad relativa del aire a 2 metros media diaria": "RH_mean",
    "Radiación solar diaria": "Radiation_Wh_m2",
    "Temperatura máxima diaria": "Tmax",
    "Temperatura mínima diaria": "Tmin",
    "Velocidad 10 minutal del viento media diaria": "Wind_speed",
}

datos_et0 = df_tibaitata_limpio[
    df_tibaitata_limpio["parametro_limpio"].isin(parametros_et0)
].copy()

duplicados_et0 = datos_et0[
    datos_et0.duplicated(["Fecha", "parametro_limpio"], keep=False)
].sort_values(["Fecha", "parametro_limpio"])

if duplicados_et0.empty:
    print("No se encontraron duplicados para la combinación Fecha-parametro_limpio.")
else:
    print("Se encontraron los siguientes registros duplicados:")
    display(duplicados_et0[["Fecha", "parametro_limpio", "Valor"]])
    raise ValueError(
        "No se puede realizar el pivotado: existen duplicados para "
        "la combinación Fecha-parametro_limpio."
    )

fechas_disponibles = df_tibaitata_limpio[["Fecha"]].drop_duplicates()

df_tibaitata_et0 = (
    datos_et0.assign(parametro_limpio=datos_et0["parametro_limpio"].map(parametros_et0))
    .pivot(index="Fecha", columns="parametro_limpio", values="Valor")
    .reindex(columns=parametros_et0.values())
    .rename_axis(None, axis=1)
    .reset_index()
)

df_tibaitata_et0 = (
    fechas_disponibles.merge(df_tibaitata_et0, on="Fecha", how="left")
    .sort_values("Fecha")
    .reset_index(drop=True)
)

print("Primeras filas:")
display(df_tibaitata_et0.head())
print("Dimensiones:", df_tibaitata_et0.shape)
print("Tipos de datos:")
display(df_tibaitata_et0.dtypes)
print("Valores nulos por columna:")
display(df_tibaitata_et0.isna().sum())

No se encontraron duplicados para la combinación Fecha-parametro_limpio.
Primeras filas:


,Fecha,RH_mean,Radiation_Wh_m2,Tmax,Tmin,Wind_speed
0,2005-02-02,89.531,1696.5,19.8,4.6,1.645
1,2005-02-03,83.958,4480.5,20.2,5.6,1.965
2,2005-02-04,89.237,2363.3,18.6,6.8,2.149
3,2005-02-05,80.562,3137.0,20.4,9.0,2.558
4,2005-02-06,89.500,1992.7,18.8,10.8,1.965


Dimensiones: (2909, 6)
Tipos de datos:


Fecha              datetime64[us]
RH_mean                   float64
Radiation_Wh_m2           float64
Tmax                      float64
Tmin                      float64
Wind_speed                float64
dtype: object

Valores nulos por columna:


Fecha              0
RH_mean            0
Radiation_Wh_m2    0
Tmax               0
Tmin               0
Wind_speed         0
dtype: int64

In [6]:
columnas_originales_et0 = df_tibaitata_et0.columns.tolist()
numero_registros_et0 = len(df_tibaitata_et0)

# El cálculo vectorizado conserva NaN cuando falta Tmax o Tmin.
df_tibaitata_et0["T_mean"] = (df_tibaitata_et0["Tmax"] + df_tibaitata_et0["Tmin"]) / 2

assert len(df_tibaitata_et0) == numero_registros_et0
assert df_tibaitata_et0.columns[:len(columnas_originales_et0)].tolist() == columnas_originales_et0
assert df_tibaitata_et0["T_mean"].isna().equals(
    df_tibaitata_et0[["Tmax", "Tmin"]].isna().any(axis=1)
)
assert (
    df_tibaitata_et0.loc[df_tibaitata_et0["T_mean"].notna(), "T_mean"]
    == (
        df_tibaitata_et0.loc[df_tibaitata_et0["T_mean"].notna(), "Tmax"]
        + df_tibaitata_et0.loc[df_tibaitata_et0["T_mean"].notna(), "Tmin"]
    ) / 2
).all()

print("Verificaciones: registros y columnas originales conservados; propagación de NaN confirmada.")
display(df_tibaitata_et0[["Tmax", "Tmin", "T_mean"]].head())
print("Valores nulos en T_mean:", df_tibaitata_et0["T_mean"].isna().sum())

Verificaciones: registros y columnas originales conservados; propagación de NaN confirmada.


,Tmax,Tmin,T_mean
0,19.8,4.6,12.2
1,20.2,5.6,12.9
2,18.6,6.8,12.7
3,20.4,9.0,14.7
4,18.8,10.8,14.8


Valores nulos en T_mean: 0


In [7]:
# Presión de vapor de saturación según FAO-56.
def presion_vapor_saturacion(temperatura):
    return 0.6108 * np.exp((17.27 * temperatura) / (temperatura + 237.3))

columnas_antes_es = df_tibaitata_et0.columns.tolist()
valores_antes_es = df_tibaitata_et0[columnas_antes_es].copy()

# Las operaciones vectorizadas conservan NaN cuando falta Tmax o Tmin.
df_tibaitata_et0["es_tmax"] = presion_vapor_saturacion(df_tibaitata_et0["Tmax"])
df_tibaitata_et0["es_tmin"] = presion_vapor_saturacion(df_tibaitata_et0["Tmin"])
df_tibaitata_et0["es"] = (df_tibaitata_et0["es_tmax"] + df_tibaitata_et0["es_tmin"]) / 2

assert df_tibaitata_et0[columnas_antes_es].equals(valores_antes_es)
assert df_tibaitata_et0["es_tmax"].isna().equals(df_tibaitata_et0["Tmax"].isna())
assert df_tibaitata_et0["es_tmin"].isna().equals(df_tibaitata_et0["Tmin"].isna())
assert np.isfinite(df_tibaitata_et0[["es_tmax", "es_tmin", "es"]].dropna()).all().all()
assert (df_tibaitata_et0[["es_tmax", "es_tmin", "es"]].dropna() > 0).all().all()
assert np.allclose(
    df_tibaitata_et0["es"].dropna(),
    ((df_tibaitata_et0["es_tmax"] + df_tibaitata_et0["es_tmin"]) / 2).dropna(),
)

print("Resultados de presión de vapor de saturación (kPa):")
display(df_tibaitata_et0[["Tmax", "Tmin", "es_tmax", "es_tmin", "es"]].head())
print("Valores nulos en las columnas nuevas:")
display(df_tibaitata_et0[["es_tmax", "es_tmin", "es"]].isna().sum())

Resultados de presión de vapor de saturación (kPa):


,Tmax,Tmin,es_tmax,es_tmin,es
0,19.8,4.6,2.309,0.848,1.579
1,20.2,5.6,2.367,0.910,1.638
2,18.6,6.8,2.143,0.988,1.566
3,20.4,9.0,2.397,1.148,1.772
4,18.8,10.8,2.170,1.295,1.733


Valores nulos en las columnas nuevas:


es_tmax    0
es_tmin    0
es         0
dtype: int64

In [8]:
# Presión de vapor de saturación y pendiente a la temperatura media.
columnas_antes_objetivo = df_tibaitata_et0.columns.tolist()
tmean = df_tibaitata_et0["T_mean"]
denominador = tmean + 237.3

with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
    es_tmean = 0.6108 * np.exp((17.27 * tmean) / denominador)
    delta = np.where(
        denominador.ne(0),
        (4098 * es_tmean) / (denominador ** 2),
        np.nan,
    )

df_tibaitata_et0["es_tmean"] = es_tmean
df_tibaitata_et0["delta"] = delta

assert df_tibaitata_et0.columns.tolist()[: len(columnas_antes_objetivo)] == columnas_antes_objetivo
assert df_tibaitata_et0["es_tmean"].isna().equals(tmean.isna())
assert df_tibaitata_et0["delta"].isna().equals(tmean.isna())
assert np.isfinite(df_tibaitata_et0[["es_tmean", "delta"]].dropna()).all().all()
assert (df_tibaitata_et0[["es_tmean", "delta"]].dropna() > 0).all().all()

print("Resultados a T_mean (es_tmean en kPa; delta en kPa/°C):")
display(df_tibaitata_et0[["T_mean", "es_tmean", "delta"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[["T_mean", "es_tmean", "delta"]].isna().sum())

Resultados a T_mean (es_tmean en kPa; delta en kPa/°C):


,T_mean,es_tmean,delta
0,12.2,1.421,0.094
1,12.9,1.488,0.097
2,12.7,1.469,0.096
3,14.7,1.673,0.108
4,14.8,1.684,0.109


Valores nulos:


T_mean      0
es_tmean    0
delta       0
dtype: int64

## Presión real de vapor

Se calcula `ea` mediante la aproximación FAO-56 basada en la humedad relativa media diaria: `ea = es * RH_mean / 100`. La humedad relativa debe estar expresada como porcentaje.

In [9]:
columnas_antes_ea = df_tibaitata_et0.columns.tolist()

assert {"es", "RH_mean"}.issubset(df_tibaitata_et0.columns)
rh_valida = df_tibaitata_et0["RH_mean"].dropna()
assert rh_valida.between(0, 100).all(), "RH_mean debe estar entre 0 y 100 %."
assert rh_valida.max() > 1, "RH_mean parece estar en escala 0-1, no en porcentaje."

df_tibaitata_et0["ea"] = (
    df_tibaitata_et0["es"] * df_tibaitata_et0["RH_mean"] / 100
)

filas_completas_ea = df_tibaitata_et0[["es", "RH_mean"]].notna().all(axis=1)
assert df_tibaitata_et0["ea"].isna().equals(~filas_completas_ea)
assert df_tibaitata_et0.columns[:len(columnas_antes_ea)].tolist() == columnas_antes_ea
assert np.isfinite(df_tibaitata_et0.loc[filas_completas_ea, "ea"]).all()
assert (
    (df_tibaitata_et0.loc[filas_completas_ea, "ea"] >= 0)
    & (df_tibaitata_et0.loc[filas_completas_ea, "ea"] <= df_tibaitata_et0.loc[filas_completas_ea, "es"])
).all()
assert np.allclose(
    df_tibaitata_et0.loc[filas_completas_ea, "ea"],
    df_tibaitata_et0.loc[filas_completas_ea, "es"]
    * df_tibaitata_et0.loc[filas_completas_ea, "RH_mean"]
    / 100,
)

print("Resultados de presión real de vapor (kPa):")
display(df_tibaitata_et0[["RH_mean", "es", "ea"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[["RH_mean", "es", "ea"]].isna().sum())
print("Verificaciones: RH_mean en porcentaje; NaN propagados; 0 <= ea <= es.")

Resultados de presión real de vapor (kPa):


,RH_mean,es,ea
0,89.531,1.579,1.414
1,83.958,1.638,1.376
2,89.237,1.566,1.397
3,80.562,1.772,1.428
4,89.500,1.733,1.551


Valores nulos:


RH_mean    0
es         0
ea         0
dtype: int64

Verificaciones: RH_mean en porcentaje; NaN propagados; 0 <= ea <= es.


## Presión atmosférica

Se calcula la presión atmosférica media de la estación con la ecuación FAO-56 usando exclusivamente la altitud de la estación Tibaitatá, leída de `df_estacion["altitud"]`.

In [10]:
assert "altitud" in df_estacion.columns
altitudes_estacion = pd.to_numeric(df_estacion["altitud"], errors="coerce").dropna().unique()
assert len(altitudes_estacion) == 1, "La estación debe tener una única altitud válida."
ALTITUD_TIBAITATA_M = altitudes_estacion[0]

assert "P" not in df_tibaitata_et0.columns, "La columna P ya existe; no se sobrescribirá."
columnas_antes_p = df_tibaitata_et0.columns.tolist()
numero_registros_antes_p = len(df_tibaitata_et0)

presion_atmosferica = 101.3 * ((293 - 0.0065 * ALTITUD_TIBAITATA_M) / 293) ** 5.26
df_tibaitata_et0["P"] = presion_atmosferica

assert len(df_tibaitata_et0) == numero_registros_antes_p
assert df_tibaitata_et0.columns[:len(columnas_antes_p)].tolist() == columnas_antes_p
assert np.isfinite(presion_atmosferica)
assert 0 < presion_atmosferica < 101.3
assert df_tibaitata_et0["P"].notna().all()
assert np.isclose(df_tibaitata_et0["P"], presion_atmosferica).all()
assert df_tibaitata_et0["P"].nunique() == 1

print(f"Presión atmosférica calculada para z = {ALTITUD_TIBAITATA_M} m: {presion_atmosferica:.6f} kPa")
print(f"Filas asignadas correctamente: {df_tibaitata_et0['P'].notna().sum()} de {len(df_tibaitata_et0)}")
display(df_tibaitata_et0[["Fecha", "P"]].head())

Presión atmosférica calculada para z = 2543 m: 74.637733 kPa
Filas asignadas correctamente: 2909 de 2909


,Fecha,P
0,2005-02-02,74.638
1,2005-02-03,74.638
2,2005-02-04,74.638
3,2005-02-05,74.638
4,2005-02-06,74.638


## Constante psicrométrica

Se calcula `gamma` (kPa/°C) con la ecuación FAO-56 `gamma = 0.000665 * P`, donde `P` es la presión atmosférica en kPa. Como `P` es constante para la estación, `gamma` también lo es. Si `P` falta en una fecha, `gamma` queda en NaN y no se imputa ningún valor.

In [11]:
assert "P" in df_tibaitata_et0.columns
assert "gamma" not in df_tibaitata_et0.columns, "La columna gamma ya existe; no se sobrescribirá."

columnas_antes_gamma = df_tibaitata_et0.columns.tolist()
valores_antes_gamma = df_tibaitata_et0[columnas_antes_gamma].copy()
fechas_antes_gamma = df_tibaitata_et0["Fecha"].copy()

# La operación vectorizada propaga NaN cuando falta P.
df_tibaitata_et0["gamma"] = 0.000665 * df_tibaitata_et0["P"]

gamma_valida = df_tibaitata_et0["gamma"].dropna()

assert df_tibaitata_et0[columnas_antes_gamma].equals(valores_antes_gamma)
assert df_tibaitata_et0.columns[:len(columnas_antes_gamma)].tolist() == columnas_antes_gamma
assert df_tibaitata_et0["Fecha"].equals(fechas_antes_gamma)
assert df_tibaitata_et0["gamma"].isna().equals(df_tibaitata_et0["P"].isna())
assert np.isfinite(gamma_valida).all()
assert (gamma_valida > 0).all()
assert gamma_valida.nunique() == 1, "gamma debe ser constante."
assert np.allclose(gamma_valida, 0.000665 * df_tibaitata_et0["P"].dropna())

valor_gamma = gamma_valida.iloc[0]
print(f"gamma = {valor_gamma:.6f} kPa/°C (P = {df_tibaitata_et0['P'].iloc[0]:.6f} kPa)")
print("Valores nulos en gamma:", df_tibaitata_et0["gamma"].isna().sum())
print("Verificaciones: columnas y fechas originales conservadas; gamma positiva y constante; NaN propagado desde P.")
display(df_tibaitata_et0[["Fecha", "P", "gamma"]].head())

gamma = 0.049634 kPa/°C (P = 74.637733 kPa)
Valores nulos en gamma: 0
Verificaciones: columnas y fechas originales conservadas; gamma positiva y constante; NaN propagado desde P.


,Fecha,P,gamma
0,2005-02-02,74.638,0.05
1,2005-02-03,74.638,0.05
2,2005-02-04,74.638,0.05
3,2005-02-05,74.638,0.05
4,2005-02-06,74.638,0.05


## Radiación solar observada

Se convierte la radiación solar diaria acumulada de Wh/m²/día a MJ/m²/día con `Rs = Radiation_Wh_m2 * 0.0036` (1 Wh = 3600 J = 0.0036 MJ). No se aplica ninguna otra conversión. Si `Radiation_Wh_m2` falta, `Rs` queda en NaN y no se imputa ningún valor.

In [12]:
assert "Radiation_Wh_m2" in df_tibaitata_et0.columns
assert "Rs" not in df_tibaitata_et0.columns, "La columna Rs ya existe; no se sobrescribirá."

columnas_antes_rs = df_tibaitata_et0.columns.tolist()
valores_antes_rs = df_tibaitata_et0[columnas_antes_rs].copy()
fechas_antes_rs = df_tibaitata_et0["Fecha"].copy()
numero_registros_antes_rs = len(df_tibaitata_et0)

WH_A_MJ = 0.0036  # MJ por Wh

# La operación vectorizada propaga NaN cuando falta la radiación original.
df_tibaitata_et0["Rs"] = df_tibaitata_et0["Radiation_Wh_m2"] * WH_A_MJ

rs_valida = df_tibaitata_et0["Rs"].dropna()

assert len(df_tibaitata_et0) == numero_registros_antes_rs
assert df_tibaitata_et0[columnas_antes_rs].equals(valores_antes_rs)
assert df_tibaitata_et0.columns[:len(columnas_antes_rs)].tolist() == columnas_antes_rs
assert df_tibaitata_et0["Fecha"].equals(fechas_antes_rs)
assert df_tibaitata_et0["Rs"].isna().equals(df_tibaitata_et0["Radiation_Wh_m2"].isna())
assert np.isfinite(rs_valida).all(), "Rs contiene valores infinitos."
assert (rs_valida >= 0).all(), "Rs contiene valores negativos."
assert np.allclose(rs_valida, df_tibaitata_et0["Radiation_Wh_m2"].dropna() * 0.0036)

# Ejemplos numéricos de referencia (1 Wh = 0.0036 MJ).
ejemplos_rs = pd.Series([0.0, 1000.0, 5000.0, 1696.5], name="Radiation_Wh_m2")
esperados_rs = pd.Series([0.0, 3.6, 18.0, 6.1074])
assert np.allclose(ejemplos_rs * WH_A_MJ, esperados_rs)

print("Ejemplos de conversión (Wh/m²/día -> MJ/m²/día):")
display(pd.DataFrame({"Radiation_Wh_m2": ejemplos_rs, "Rs_esperado": esperados_rs}))
print("Primeras filas del dataset:")
display(df_tibaitata_et0[["Fecha", "Radiation_Wh_m2", "Rs"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[["Radiation_Wh_m2", "Rs"]].isna().sum())
print("Resumen de Rs (MJ/m²/día):")
display(rs_valida.describe())
print("Verificaciones: columnas y fechas originales conservadas; NaN propagados; sin infinitos ni negativos.")

Ejemplos de conversión (Wh/m²/día -> MJ/m²/día):


,Radiation_Wh_m2,Rs_esperado
0,0.0,0.000
1,1000.0,3.600
2,5000.0,18.000
3,1696.5,6.107


Primeras filas del dataset:


,Fecha,Radiation_Wh_m2,Rs
0,2005-02-02,1696.5,6.107
1,2005-02-03,4480.5,16.130
2,2005-02-04,2363.3,8.508
3,2005-02-05,3137.0,11.293
4,2005-02-06,1992.7,7.174


Valores nulos:


Radiation_Wh_m2    0
Rs                 0
dtype: int64

Resumen de Rs (MJ/m²/día):


count    2909.000
mean       14.962
std         4.469
min         0.839
25%        11.664
50%        14.596
75%        18.119
max        26.862
Name: Rs, dtype: float64

Verificaciones: columnas y fechas originales conservadas; NaN propagados; sin infinitos ni negativos.


## Variables astronómicas

Se calculan con las ecuaciones FAO-56 las variables que dependen de la fecha y de la latitud de Tibaitatá. La latitud se lee de `df_estacion["latitud"]` (grados decimales, norte positivo):

- `J`: día del año (`dayofyear` de `Fecha`).
- `phi_rad`: latitud en radianes, `phi = latitud * pi / 180`.
- `dr`: distancia relativa inversa Tierra-Sol, `dr = 1 + 0.033 cos(2 pi J / 365)`.
- `solar_declination`: declinación solar en radianes, `delta = 0.409 sin(2 pi J / 365 - 1.39)`.
- `omega_s`: ángulo horario al atardecer en radianes, `omega_s = arccos(-tan(phi) tan(delta))`. El argumento del arcocoseno se limita a [-1, 1] para evitar errores numéricos.

**Tratamiento de años bisiestos.** FAO-56 usa 365 días en las ecuaciones de `dr` y `delta`. Aquí `J` es el `dayofyear` calendario sin ajustes: en años bisiestos, las fechas posteriores al 29 de febrero tienen `J` una unidad mayor que en un año común, y el 31 de diciembre toma `J = 366`. No se elimina ni se reasigna ninguna fecha. El desfase equivale a un día de 365 (aprox. 0.017 rad en el argumento angular), por lo que el efecto sobre `dr`, `delta` y `omega_s` es mínimo. Como la estación está cerca del ecuador, el argumento del arcocoseno queda muy por debajo de 1 en valor absoluto y el límite no debería activarse.

In [13]:
assert "Fecha" in df_tibaitata_et0.columns
columnas_astronomicas = ["J", "phi_rad", "dr", "solar_declination", "omega_s"]
assert not set(columnas_astronomicas) & set(df_tibaitata_et0.columns), "Alguna columna astronómica ya existe; no se sobrescribirá."

columnas_antes_astro = df_tibaitata_et0.columns.tolist()
valores_antes_astro = df_tibaitata_et0[columnas_antes_astro].copy()
numero_registros_antes_astro = len(df_tibaitata_et0)
forma_limpio_antes_astro = df_tibaitata_limpio.shape  # df_tibaitata_limpio no se modifica

# La latitud se toma de df_estacion (columna "latitud", grados decimales; norte positivo).
assert "latitud" in df_estacion.columns
latitudes_estacion = pd.to_numeric(df_estacion["latitud"], errors="coerce").dropna().unique()
assert len(latitudes_estacion) == 1, "La estación debe tener una única latitud válida."
LATITUD_TIBAITATA_DEG = float(latitudes_estacion[0])
assert -90 < LATITUD_TIBAITATA_DEG < 90, "La latitud debe estar en grados entre -90 y 90."

fechas_astro = pd.to_datetime(df_tibaitata_et0["Fecha"])
assert fechas_astro.notna().all(), "Existen fechas faltantes en Fecha."

# Día del año calendario (1-366); en años bisiestos el 31-dic toma J = 366.
J = fechas_astro.dt.dayofyear
phi_rad = np.deg2rad(LATITUD_TIBAITATA_DEG)
angulo_anual = 2 * np.pi * J / 365  # convención FAO-56: 365 días

dr = 1 + 0.033 * np.cos(angulo_anual)
solar_declination = 0.409 * np.sin(angulo_anual - 1.39)

argumento_arccos = -np.tan(phi_rad) * np.tan(solar_declination)
n_argumento_fuera_rango = int(((argumento_arccos < -1) | (argumento_arccos > 1)).sum())
omega_s = np.arccos(np.clip(argumento_arccos, -1, 1))

df_tibaitata_et0["J"] = J
df_tibaitata_et0["phi_rad"] = phi_rad
df_tibaitata_et0["dr"] = dr
df_tibaitata_et0["solar_declination"] = solar_declination
df_tibaitata_et0["omega_s"] = omega_s

# Verificaciones: datos originales intactos.
assert len(df_tibaitata_et0) == numero_registros_antes_astro
assert df_tibaitata_et0[columnas_antes_astro].equals(valores_antes_astro)
assert df_tibaitata_et0.columns[:len(columnas_antes_astro)].tolist() == columnas_antes_astro
assert df_tibaitata_limpio.shape == forma_limpio_antes_astro

# Verificaciones: validez numérica y unidades (radianes).
assert df_tibaitata_et0[columnas_astronomicas].notna().all().all()
assert np.isfinite(df_tibaitata_et0[columnas_astronomicas]).all().all()
assert df_tibaitata_et0["J"].between(1, 366).all()
assert np.isclose(df_tibaitata_et0["phi_rad"], LATITUD_TIBAITATA_DEG * np.pi / 180).all()
assert abs(phi_rad) < np.pi / 2
assert df_tibaitata_et0["dr"].between(1 - 0.033, 1 + 0.033).all()
assert df_tibaitata_et0["solar_declination"].abs().max() <= 0.409 + 1e-12  # rad (~23.45°)
assert df_tibaitata_et0["omega_s"].between(0, np.pi).all()
assert df_tibaitata_et0["omega_s"].max() < 2  # rad; en grados sería > 2 con facilidad

es_bisiesto = fechas_astro.dt.is_leap_year
print("Registros:", len(df_tibaitata_et0))
print("Registros en años bisiestos:", int(es_bisiesto.sum()), "| con J = 366:", int((J == 366).sum()))
print("Argumentos de arccos fuera de [-1, 1] antes de limitar:", n_argumento_fuera_rango)
print(f"phi_rad = {phi_rad:.6f} rad ({LATITUD_TIBAITATA_DEG}°)")
print("Verificaciones: datos originales intactos; valores finitos; ángulos en radianes.")
display(df_tibaitata_et0[["Fecha"] + columnas_astronomicas].head())
display(df_tibaitata_et0[columnas_astronomicas].describe().loc[["min", "mean", "max"]])
print("Valores nulos:")
display(df_tibaitata_et0[columnas_astronomicas].isna().sum())

Registros: 2909
Registros en años bisiestos: 637 | con J = 366: 1
Argumentos de arccos fuera de [-1, 1] antes de limitar: 0
phi_rad = 0.081833 rad (4.688662°)
Verificaciones: datos originales intactos; valores finitos; ángulos en radianes.


,Fecha,J,phi_rad,dr,solar_declination,omega_s
0,2005-02-02,33,0.082,1.028,-0.300,1.545
1,2005-02-03,34,0.082,1.028,-0.295,1.546
2,2005-02-04,35,0.082,1.027,-0.290,1.546
3,2005-02-05,36,0.082,1.027,-0.285,1.547
4,2005-02-06,37,0.082,1.027,-0.280,1.547


,J,phi_rad,dr,solar_declination,omega_s
min,1.000,0.082,0.967,-0.409,1.535
mean,181.858,0.082,1.001,-0.008,1.570
max,366.000,0.082,1.033,0.409,1.606


Valores nulos:


J                    0
phi_rad              0
dr                   0
solar_declination    0
omega_s              0
dtype: int64

## Radiación extraterrestre

Se calcula `Ra` (MJ/m²/día) con la ecuación FAO-56 usando las variables astronómicas ya calculadas (`dr`, `omega_s`, `phi_rad`, `solar_declination`) y la constante `SOLAR_CONSTANT` (`Gsc = 0.0820` MJ/m²/min):

`Ra = (24 * 60 / pi) * Gsc * dr * [omega_s * sin(phi) * sin(delta) + cos(phi) * cos(delta) * sin(omega_s)]`

Todos los ángulos están en radianes. Si falta alguna variable de entrada, `Ra` queda en NaN y no se imputa ningún valor.

In [14]:
columnas_ra_requeridas = ["dr", "omega_s", "phi_rad", "solar_declination"]
assert set(columnas_ra_requeridas).issubset(df_tibaitata_et0.columns)
assert "Ra" not in df_tibaitata_et0.columns, "La columna Ra ya existe; no se sobrescribirá."
assert SOLAR_CONSTANT == 0.0820

columnas_antes_ra = df_tibaitata_et0.columns.tolist()
valores_antes_ra = df_tibaitata_et0[columnas_antes_ra].copy()
numero_registros_antes_ra = len(df_tibaitata_et0)
forma_limpio_antes_ra = df_tibaitata_limpio.shape

dr_ra = df_tibaitata_et0["dr"]
omega_s_ra = df_tibaitata_et0["omega_s"]
phi_ra = df_tibaitata_et0["phi_rad"]
delta_ra = df_tibaitata_et0["solar_declination"]

# Las operaciones vectorizadas propagan NaN si falta alguna variable de entrada.
termino_angular = (
    omega_s_ra * np.sin(phi_ra) * np.sin(delta_ra)
    + np.cos(phi_ra) * np.cos(delta_ra) * np.sin(omega_s_ra)
)
df_tibaitata_et0["Ra"] = (24 * 60 / np.pi) * SOLAR_CONSTANT * dr_ra * termino_angular

ra_valida = df_tibaitata_et0["Ra"].dropna()
filas_completas_ra = df_tibaitata_et0[columnas_ra_requeridas].notna().all(axis=1)

# Datos originales intactos.
assert len(df_tibaitata_et0) == numero_registros_antes_ra
assert df_tibaitata_et0[columnas_antes_ra].equals(valores_antes_ra)
assert df_tibaitata_et0.columns[:len(columnas_antes_ra)].tolist() == columnas_antes_ra
assert df_tibaitata_limpio.shape == forma_limpio_antes_ra

# NaN propagados, valores finitos y positivos.
assert df_tibaitata_et0["Ra"].isna().equals(~filas_completas_ra)
assert np.isfinite(ra_valida).all(), "Ra contiene valores infinitos."
assert (ra_valida > 0).all(), "Ra contiene valores no positivos."

# Rango físico: en latitud ~4.7° N, Ra anual ronda 30-38 MJ/m²/día.
assert ra_valida.between(25, 42).all(), "Ra fuera del rango físico esperado para esta latitud."

# Ra debe ser mayor que Rs observada (la atmósfera solo atenúa la radiación).
rs_comparable = df_tibaitata_et0.loc[filas_completas_ra, ["Rs", "Ra"]].dropna()
n_rs_mayor_ra = int((rs_comparable["Rs"] > rs_comparable["Ra"]).sum())

print("Ra calculada (MJ/m²/día).")
print("Primeras filas:")
display(df_tibaitata_et0[["Fecha", "J", "dr", "omega_s", "solar_declination", "Ra"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[columnas_ra_requeridas + ["Ra"]].isna().sum())
print("Resumen de Ra (MJ/m²/día):")
display(ra_valida.describe())
print("Ra media por mes (MJ/m²/día):")
display(df_tibaitata_et0.groupby(df_tibaitata_et0["Fecha"].dt.month)["Ra"].mean().round(2).to_frame("Ra_media"))
print(f"Registros con Rs > Ra: {n_rs_mayor_ra} de {len(rs_comparable)} (informativo; no se modifica nada).")
print("Verificaciones: columnas y registros originales conservados; NaN propagados; Ra positiva y en rango físico.")

Ra calculada (MJ/m²/día).
Primeras filas:


,Fecha,J,dr,omega_s,solar_declination,Ra
0,2005-02-02,33,1.028,1.545,-0.300,35.335
1,2005-02-03,34,1.028,1.546,-0.295,35.402
2,2005-02-04,35,1.027,1.546,-0.290,35.468
3,2005-02-05,36,1.027,1.547,-0.285,35.535
4,2005-02-06,37,1.027,1.547,-0.280,35.602


Valores nulos:


dr                   0
omega_s              0
phi_rad              0
solar_declination    0
Ra                   0
dtype: int64

Resumen de Ra (MJ/m²/día):


count    2909.000
mean       35.923
std         1.258
min        33.524
25%        35.139
50%        36.067
75%        37.057
max        37.727
Name: Ra, dtype: float64

Ra media por mes (MJ/m²/día):


,Ra_media
Fecha,
1,34.38
2,36.15
3,37.50
4,37.38
5,36.17
6,35.23
7,35.54
8,36.68
9,37.21


Registros con Rs > Ra: 0 de 2909 (informativo; no se modifica nada).
Verificaciones: columnas y registros originales conservados; NaN propagados; Ra positiva y en rango físico.


## Radiación de cielo despejado

Se calcula `Rso` (MJ/m²/día) con la ecuación FAO-56 `Rso = (0.75 + 2e-5 * z) * Ra`, donde `z` es la altitud de la estación, leída de `df_estacion["altitud"]` y `Ra` la radiación extraterrestre. Si `Ra` falta, `Rso` queda en NaN y no se imputa ningún valor.

In [15]:
assert "Ra" in df_tibaitata_et0.columns
assert "Rso" not in df_tibaitata_et0.columns, "La columna Rso ya existe; no se sobrescribirá."

ALTITUD_RSO_M = ALTITUD_TIBAITATA_M  # m, leída de df_estacion["altitud"]

columnas_antes_rso = df_tibaitata_et0.columns.tolist()
valores_antes_rso = df_tibaitata_et0[columnas_antes_rso].copy()
numero_registros_antes_rso = len(df_tibaitata_et0)
forma_limpio_antes_rso = df_tibaitata_limpio.shape

# Coeficiente de altitud: 0.75 + 2e-5 * z, con z de df_estacion.
coeficiente_rso = 0.75 + 2e-5 * ALTITUD_RSO_M
assert np.isclose(coeficiente_rso, 0.75 + 2e-5 * float(ALTITUD_TIBAITATA_M))
assert 0.75 < coeficiente_rso < 1.0

# La operación vectorizada propaga NaN cuando falta Ra.
df_tibaitata_et0["Rso"] = coeficiente_rso * df_tibaitata_et0["Ra"]

rso_valida = df_tibaitata_et0["Rso"].dropna()
ra_con_rso = df_tibaitata_et0.loc[df_tibaitata_et0["Rso"].notna(), "Ra"]

# Datos originales intactos.
assert len(df_tibaitata_et0) == numero_registros_antes_rso
assert df_tibaitata_et0[columnas_antes_rso].equals(valores_antes_rso)
assert df_tibaitata_et0.columns[:len(columnas_antes_rso)].tolist() == columnas_antes_rso
assert df_tibaitata_limpio.shape == forma_limpio_antes_rso

# NaN propagados, valores finitos y positivos.
assert df_tibaitata_et0["Rso"].isna().equals(df_tibaitata_et0["Ra"].isna())
assert np.isfinite(rso_valida).all(), "Rso contiene valores infinitos."
assert (rso_valida > 0).all(), "Rso contiene valores no positivos."

# Coherencia con Ra: Rso < Ra y Rso / Ra constante e igual al coeficiente.
assert (rso_valida < ra_con_rso).all(), "Rso debe ser menor que Ra."
assert np.allclose(rso_valida / ra_con_rso, coeficiente_rso)
assert np.allclose(rso_valida, (0.75 + 2e-5 * float(ALTITUD_TIBAITATA_M)) * ra_con_rso)

# Registros con Rs > Rso (informativo; la relación Rs/Rso se usa más adelante).
comparable_rso = df_tibaitata_et0[["Rs", "Rso"]].dropna()
n_rs_mayor_rso = int((comparable_rso["Rs"] > comparable_rso["Rso"]).sum())

print(f"Altitud: {ALTITUD_RSO_M} m | coeficiente (0.75 + 2e-5 * z) = {coeficiente_rso:.5f}")
print("Primeras filas:")
display(df_tibaitata_et0[["Fecha", "Ra", "Rso"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[["Ra", "Rso"]].isna().sum())
print("Resumen de Rso (MJ/m²/día):")
display(rso_valida.describe())
print(f"Razón Rso/Ra: mín = {(rso_valida / ra_con_rso).min():.5f}, máx = {(rso_valida / ra_con_rso).max():.5f}")
print(f"Registros con Rs > Rso: {n_rs_mayor_rso} de {len(comparable_rso)} (informativo; no se modifica nada).")
print("Verificaciones: columnas y registros originales conservados; NaN propagados; Rso positiva y menor que Ra.")

Altitud: 2543 m | coeficiente (0.75 + 2e-5 * z) = 0.80086
Primeras filas:


,Fecha,Ra,Rso
0,2005-02-02,35.335,28.299
1,2005-02-03,35.402,28.352
2,2005-02-04,35.468,28.405
3,2005-02-05,35.535,28.459
4,2005-02-06,35.602,28.512


Valores nulos:


Ra     0
Rso    0
dtype: int64

Resumen de Rso (MJ/m²/día):


count    2909.000
mean       28.770
std         1.007
min        26.848
25%        28.141
50%        28.884
75%        29.678
max        30.214
Name: Rso, dtype: float64

Razón Rso/Ra: mín = 0.80086, máx = 0.80086
Registros con Rs > Rso: 0 de 2909 (informativo; no se modifica nada).
Verificaciones: columnas y registros originales conservados; NaN propagados; Rso positiva y menor que Ra.


## Radiación neta de onda corta

Se calcula `Rns` (MJ/m²/día) con la ecuación FAO-56 `Rns = (1 - alpha) * Rs`, usando el albedo FAO-56 del cultivo de referencia `alpha = 0.23`, es decir, `Rns = 0.77 * Rs`. `Rs` no se modifica. Si `Rs` falta, `Rns` queda en NaN y no se imputa ningún valor.

In [16]:
assert "Rs" in df_tibaitata_et0.columns
assert "Rns" not in df_tibaitata_et0.columns, "La columna Rns ya existe; no se sobrescribirá."

ALBEDO_FAO56 = 0.23  # adimensional, cultivo de referencia

columnas_antes_rns = df_tibaitata_et0.columns.tolist()
valores_antes_rns = df_tibaitata_et0[columnas_antes_rns].copy()
numero_registros_antes_rns = len(df_tibaitata_et0)
forma_limpio_antes_rns = df_tibaitata_limpio.shape

# La operación vectorizada propaga NaN cuando falta Rs.
df_tibaitata_et0["Rns"] = (1 - ALBEDO_FAO56) * df_tibaitata_et0["Rs"]

rs_con_rns = df_tibaitata_et0["Rs"].dropna()
rns_valida = df_tibaitata_et0["Rns"].dropna()

# Datos originales intactos (incluida Rs); sin eliminar fechas.
assert len(df_tibaitata_et0) == numero_registros_antes_rns
assert df_tibaitata_et0[columnas_antes_rns].equals(valores_antes_rns)
assert df_tibaitata_et0.columns[:len(columnas_antes_rns)].tolist() == columnas_antes_rns
assert df_tibaitata_limpio.shape == forma_limpio_antes_rns

# NaN propagados, valores finitos y no negativos.
assert df_tibaitata_et0["Rns"].isna().equals(df_tibaitata_et0["Rs"].isna())
assert np.isfinite(rns_valida).all(), "Rns contiene valores infinitos."
assert (rns_valida >= 0).all(), "Rns contiene valores negativos."

# Rns es el 77 % de Rs.
assert np.isclose(ALBEDO_FAO56, 0.23)
assert np.allclose(rns_valida, 0.77 * rs_con_rns)
assert np.allclose(rns_valida / rs_con_rns.loc[rs_con_rns > 0], 0.77)
assert (rns_valida <= rs_con_rns).all()

print(f"Albedo = {ALBEDO_FAO56} | Rns = {1 - ALBEDO_FAO56:.2f} * Rs")
print("Primeras filas:")
display(df_tibaitata_et0[["Fecha", "Rs", "Rns"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[["Rs", "Rns"]].isna().sum())
print("Resumen de Rns (MJ/m²/día):")
display(rns_valida.describe())
print(f"Razón Rns/Rs (Rs > 0): mín = {(rns_valida / rs_con_rns.loc[rs_con_rns > 0]).min():.5f}, máx = {(rns_valida / rs_con_rns.loc[rs_con_rns > 0]).max():.5f}")
print("Verificaciones: columnas y registros originales conservados; Rs sin cambios; NaN propagados; Rns = 77 % de Rs.")

Albedo = 0.23 | Rns = 0.77 * Rs
Primeras filas:


,Fecha,Rs,Rns
0,2005-02-02,6.107,4.703
1,2005-02-03,16.130,12.420
2,2005-02-04,8.508,6.551
3,2005-02-05,11.293,8.696
4,2005-02-06,7.174,5.524


Valores nulos:


Rs     0
Rns    0
dtype: int64

Resumen de Rns (MJ/m²/día):


count    2909.000
mean       11.521
std         3.441
min         0.646
25%         8.981
50%        11.239
75%        13.952
max        20.684
Name: Rns, dtype: float64

Razón Rns/Rs (Rs > 0): mín = 0.77000, máx = 0.77000
Verificaciones: columnas y registros originales conservados; Rs sin cambios; NaN propagados; Rns = 77 % de Rs.


## Radiación neta de onda larga

Se calcula `Rnl` (MJ/m²/día) con la ecuación FAO-56:

`Rnl = sigma * (Tmax_K^4 + Tmin_K^4) / 2 * (0.34 - 0.14 * sqrt(ea)) * (1.35 * Rs/Rso - 0.35)`

- `Tmax_K = Tmax + 273.16` y `Tmin_K = Tmin + 273.16` (K). Se usa 273.16, el valor indicado para esta ecuación, en lugar de `KELVIN_OFFSET` (273.15), que no se modifica.
- `sigma = STEFAN_BOLTZMANN = 4.903e-9` MJ K⁻⁴ m⁻² día⁻¹.
- `Rs`, `Rso` y `Rnl` están en MJ/m²/día; `ea` en kPa.
- `Rs_Rso_sin_limitar` guarda el cociente original y `Rs_Rso` lo limita a un máximo de 1 (FAO-56).
- El cociente queda en NaN si `Rso` no es finita o es <= 0, o si `Rs` no es finita o es negativa. `ea_sqrt` queda en NaN si `ea` es negativa o no finita. Estos casos no se corrigen: se cuentan y se muestran.
- Si falta cualquier variable, `Rnl` queda en NaN y no se imputa ningún valor.
- `Rn` no se calcula en este paso.

In [17]:
columnas_rnl_requeridas = ["Tmax", "Tmin", "ea", "Rs", "Rso"]
columnas_rnl_nuevas = ["Tmax_K", "Tmin_K", "ea_sqrt", "Rs_Rso_sin_limitar", "Rs_Rso", "Rnl"]
assert set(columnas_rnl_requeridas).issubset(df_tibaitata_et0.columns)
assert not set(columnas_rnl_nuevas) & set(df_tibaitata_et0.columns), "Alguna columna de Rnl ya existe; no se sobrescribirá."

KELVIN_OFFSET_RNL = 273.16  # K por °C, valor indicado para la ecuación de Rnl

def calcular_rnl(df):
    """Devuelve las columnas intermedias y Rnl (MJ/m²/día) sin modificar `df`."""
    tmax_k = df["Tmax"] + KELVIN_OFFSET_RNL
    tmin_k = df["Tmin"] + KELVIN_OFFSET_RNL

    # Raíz cuadrada solo si ea es finita y no negativa; en otro caso queda NaN.
    ea_valida = np.isfinite(df["ea"]) & (df["ea"] >= 0)
    ea_sqrt = np.sqrt(df["ea"].where(ea_valida))

    # Cociente solo si Rso es finita y > 0, y Rs es finita y >= 0; en otro caso queda NaN.
    razon_valida = (
        np.isfinite(df["Rso"]) & (df["Rso"] > 0) & np.isfinite(df["Rs"]) & (df["Rs"] >= 0)
    )
    rs_rso_sin_limitar = df["Rs"].where(razon_valida) / df["Rso"].where(razon_valida)
    rs_rso = rs_rso_sin_limitar.clip(upper=1.0)  # FAO-56: Rs/Rso <= 1 (NaN se conserva)

    rnl = (
        STEFAN_BOLTZMANN
        * (tmax_k ** 4 + tmin_k ** 4) / 2
        * (0.34 - 0.14 * ea_sqrt)
        * (1.35 * rs_rso - 0.35)
    )
    return pd.DataFrame(
        {
            "Tmax_K": tmax_k,
            "Tmin_K": tmin_k,
            "ea_sqrt": ea_sqrt,
            "Rs_Rso_sin_limitar": rs_rso_sin_limitar,
            "Rs_Rso": rs_rso,
            "Rnl": rnl,
        }
    )

# --- Verificación con ejemplos independientes (cálculo escalar con math) ---
import math

ejemplos_rnl = pd.DataFrame(
    {
        "Tmax": [25.0, 25.0, 25.0, 25.0, 25.0, np.nan, 25.0, 25.0],
        "Tmin": [15.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0, 15.0],
        "ea": [1.5, 1.5, 1.5, -0.1, 1.5, 1.5, np.nan, 0.0],
        "Rs": [14.0, 30.0, 14.0, 14.0, 14.0, 14.0, 14.0, 14.0],
        "Rso": [20.0, 28.0, 0.0, 20.0, np.nan, 20.0, 20.0, 20.0],
    },
    index=[
        "normal", "Rs > Rso (se limita a 1)", "Rso = 0", "ea < 0",
        "Rso NaN", "Tmax NaN", "ea NaN", "ea = 0",
    ],
)

def rnl_escalar(tmax, tmin, ea, rs, rso):
    razon = min(rs / rso, 1.0)
    return (
        4.903e-9
        * (((tmax + 273.16) ** 4) + ((tmin + 273.16) ** 4)) / 2
        * (0.34 - 0.14 * math.sqrt(ea))
        * (1.35 * razon - 0.35)
    )

resultado_ejemplos = calcular_rnl(ejemplos_rnl)
esperado_ejemplos = {
    "normal": rnl_escalar(25.0, 15.0, 1.5, 14.0, 20.0),
    "Rs > Rso (se limita a 1)": rnl_escalar(25.0, 15.0, 1.5, 30.0, 28.0),
    "ea = 0": rnl_escalar(25.0, 15.0, 0.0, 14.0, 20.0),
}
for caso, valor in esperado_ejemplos.items():
    assert math.isclose(resultado_ejemplos.loc[caso, "Rnl"], valor, rel_tol=1e-12), caso

assert resultado_ejemplos.loc["Rs > Rso (se limita a 1)", "Rs_Rso_sin_limitar"] > 1
assert resultado_ejemplos.loc["Rs > Rso (se limita a 1)", "Rs_Rso"] == 1.0
for caso in ["Rso = 0", "ea < 0", "Rso NaN", "Tmax NaN", "ea NaN"]:
    assert math.isnan(resultado_ejemplos.loc[caso, "Rnl"]), f"{caso} debe producir NaN."
assert resultado_ejemplos.loc["Rso = 0", ["Rs_Rso_sin_limitar", "Rs_Rso"]].isna().all()
assert math.isnan(resultado_ejemplos.loc["ea < 0", "ea_sqrt"])

print("Ejemplos de verificación (Rnl en MJ/m²/día):")
display(pd.concat([ejemplos_rnl, resultado_ejemplos], axis=1))
print("Ejemplos verificados contra un cálculo escalar independiente.")

# --- Cálculo sobre df_tibaitata_et0 ---
columnas_antes_rnl = df_tibaitata_et0.columns.tolist()
valores_antes_rnl = df_tibaitata_et0[columnas_antes_rnl].copy()
numero_registros_antes_rnl = len(df_tibaitata_et0)
forma_limpio_antes_rnl = df_tibaitata_limpio.shape

resultado_rnl = calcular_rnl(df_tibaitata_et0)
for columna in columnas_rnl_nuevas:
    df_tibaitata_et0[columna] = resultado_rnl[columna]

# Entradas presentes pero no válidas (no se corrigen; solo se reportan).
n_rso_no_valido = int((df_tibaitata_et0["Rso"].notna() & ~(df_tibaitata_et0["Rso"] > 0)).sum())
n_rs_negativa = int((df_tibaitata_et0["Rs"] < 0).sum())
n_ea_negativa = int((df_tibaitata_et0["ea"] < 0).sum())
n_razon_limitada = int((df_tibaitata_et0["Rs_Rso_sin_limitar"] > 1).sum())

# Datos originales y cálculos anteriores intactos; sin eliminar fechas.
assert len(df_tibaitata_et0) == numero_registros_antes_rnl
assert df_tibaitata_et0[columnas_antes_rnl].equals(valores_antes_rnl)
assert df_tibaitata_et0.columns[:len(columnas_antes_rnl)].tolist() == columnas_antes_rnl
assert df_tibaitata_limpio.shape == forma_limpio_antes_rnl

# NaN propagados desde las entradas (sin imputación).
entradas_rnl_validas = df_tibaitata_et0[["Tmax_K", "Tmin_K", "ea_sqrt", "Rs_Rso"]].notna().all(axis=1)
assert df_tibaitata_et0["Rnl"].isna().equals(~entradas_rnl_validas)
assert df_tibaitata_et0["Tmax_K"].isna().equals(df_tibaitata_et0["Tmax"].isna())
assert df_tibaitata_et0["Tmin_K"].isna().equals(df_tibaitata_et0["Tmin"].isna())

# Valores finitos y razón limitada correctamente.
rnl_valida = df_tibaitata_et0["Rnl"].dropna()
assert np.isfinite(rnl_valida).all(), "Rnl contiene valores infinitos."
assert np.isfinite(df_tibaitata_et0[["Tmax_K", "Tmin_K", "ea_sqrt", "Rs_Rso"]].dropna()).all().all()
assert (df_tibaitata_et0["Rs_Rso"].dropna() <= 1).all()
assert np.allclose(
    df_tibaitata_et0["Rs_Rso"].dropna(),
    df_tibaitata_et0["Rs_Rso_sin_limitar"].dropna().clip(upper=1),
)

# Recalculo independiente de Rnl sobre las filas válidas.
fv = df_tibaitata_et0.loc[entradas_rnl_validas]
rnl_control = (
    4.903e-9
    * ((fv["Tmax"] + 273.16) ** 4 + (fv["Tmin"] + 273.16) ** 4) / 2
    * (0.34 - 0.14 * np.sqrt(fv["ea"]))
    * (1.35 * np.minimum(fv["Rs"] / fv["Rso"], 1.0) - 0.35)
)
assert np.allclose(fv["Rnl"], rnl_control)

print(f"Registros: {len(df_tibaitata_et0)} | Rnl calculada en {len(rnl_valida)}")
print(f"Rso no válida (<= 0): {n_rso_no_valido} | Rs negativa: {n_rs_negativa} | ea negativa: {n_ea_negativa}")
print(f"Razones Rs/Rso limitadas a 1: {n_razon_limitada}")
print(f"Registros con Rnl < 0 (informativo): {int((rnl_valida < 0).sum())}")
print("Primeras filas:")
display(df_tibaitata_et0[["Fecha", "Tmax", "Tmin", "ea", "Rs", "Rso"] + columnas_rnl_nuevas].head())
print("Valores nulos en las columnas nuevas:")
display(df_tibaitata_et0[columnas_rnl_nuevas].isna().sum())
print("Resumen de Rnl (MJ/m²/día):")
display(rnl_valida.describe())
print("Verificaciones: columnas y registros originales conservados; NaN propagados; Rs/Rso <= 1; Rnl coincide con un recálculo independiente.")

Ejemplos de verificación (Rnl en MJ/m²/día):


,Tmax,Tmin,ea,Rs,Rso,Tmax_K,Tmin_K,ea_sqrt,Rs_Rso_sin_limitar,Rs_Rso,Rnl
normal,25.0,15.0,1.5,14.0,20.0,298.16,288.16,1.225,0.700,0.7,3.638
Rs > Rso (se limita a 1),25.0,15.0,1.5,30.0,28.0,298.16,288.16,1.225,1.071,1.0,6.114
Rso = 0,25.0,15.0,1.5,14.0,0.0,298.16,288.16,1.225,NaN,NaN,NaN
ea < 0,25.0,15.0,-0.1,14.0,20.0,298.16,288.16,NaN,0.700,0.7,NaN
Rso NaN,25.0,15.0,1.5,14.0,NaN,298.16,288.16,1.225,NaN,NaN,NaN
Tmax NaN,NaN,15.0,1.5,14.0,20.0,NaN,288.16,1.225,0.700,0.7,NaN
ea NaN,25.0,15.0,NaN,14.0,20.0,298.16,288.16,NaN,0.700,0.7,NaN
ea = 0,25.0,15.0,0.0,14.0,20.0,298.16,288.16,0.000,0.700,0.7,7.339


Ejemplos verificados contra un cálculo escalar independiente.


Registros: 2909 | Rnl calculada en 2909
Rso no válida (<= 0): 0 | Rs negativa: 0 | ea negativa: 0
Razones Rs/Rso limitadas a 1: 0
Registros con Rnl < 0 (informativo): 94
Primeras filas:


,Fecha,Tmax,Tmin,ea,Rs,Rso,Tmax_K,Tmin_K,ea_sqrt,Rs_Rso_sin_limitar,Rs_Rso,Rnl
0,2005-02-02,19.8,4.6,1.414,6.107,28.299,292.96,277.76,1.189,0.216,0.216,-0.332
1,2005-02-03,20.2,5.6,1.376,16.130,28.352,293.36,278.76,1.173,0.569,0.569,2.422
2,2005-02-04,18.6,6.8,1.397,8.508,28.405,291.76,279.96,1.182,0.300,0.300,0.311
3,2005-02-05,20.4,9.0,1.428,11.293,28.459,293.56,282.16,1.195,0.397,0.397,1.082
4,2005-02-06,18.8,10.8,1.551,7.174,28.512,291.96,283.96,1.245,0.252,0.252,-0.058


Valores nulos en las columnas nuevas:


Tmax_K                0
Tmin_K                0
ea_sqrt               0
Rs_Rso_sin_limitar    0
Rs_Rso                0
Rnl                   0
dtype: int64

Resumen de Rnl (MJ/m²/día):


count    2909.000
mean        2.065
std         1.256
min        -1.828
25%         1.138
50%         1.938
75%         2.908
max         6.099
Name: Rnl, dtype: float64

Verificaciones: columnas y registros originales conservados; NaN propagados; Rs/Rso <= 1; Rnl coincide con un recálculo independiente.


## Radiación neta total

Se calcula `Rn` (MJ/m²/día) como la diferencia entre la radiación neta de onda corta y la de onda larga: `Rn = Rns - Rnl`. `Rns` y `Rnl` no se modifican. Si falta alguna de las dos, `Rn` queda en NaN y no se imputa ningún valor. Los valores negativos de `Rn` se conservan sin forzarlos a cero. No se calcula ET₀ en este paso.

In [18]:
assert {"Rns", "Rnl"}.issubset(df_tibaitata_et0.columns)
assert "Rn" not in df_tibaitata_et0.columns, "La columna Rn ya existe; no se sobrescribirá."

columnas_antes_rn = df_tibaitata_et0.columns.tolist()
valores_antes_rn = df_tibaitata_et0[columnas_antes_rn].copy()
fechas_antes_rn = df_tibaitata_et0["Fecha"].copy()
numero_registros_antes_rn = len(df_tibaitata_et0)
forma_limpio_antes_rn = df_tibaitata_limpio.shape

# La resta vectorizada propaga NaN cuando falta Rns o Rnl.
df_tibaitata_et0["Rn"] = df_tibaitata_et0["Rns"] - df_tibaitata_et0["Rnl"]

filas_completas_rn = df_tibaitata_et0[["Rns", "Rnl"]].notna().all(axis=1)
rn_valida = df_tibaitata_et0["Rn"].dropna()

# Datos originales (incluidas Rns y Rnl) intactos; sin eliminar fechas.
assert len(df_tibaitata_et0) == numero_registros_antes_rn
assert df_tibaitata_et0[columnas_antes_rn].equals(valores_antes_rn)
assert df_tibaitata_et0.columns[:len(columnas_antes_rn)].tolist() == columnas_antes_rn
assert df_tibaitata_et0["Fecha"].equals(fechas_antes_rn)
assert df_tibaitata_limpio.shape == forma_limpio_antes_rn

# NaN propagados y valores finitos.
assert df_tibaitata_et0["Rn"].isna().equals(~filas_completas_rn)
assert np.isfinite(rn_valida).all(), "Rn contiene valores infinitos."

# Rn coincide con Rns - Rnl y con Rns = Rn + Rnl.
assert np.allclose(
    rn_valida,
    df_tibaitata_et0.loc[filas_completas_rn, "Rns"] - df_tibaitata_et0.loc[filas_completas_rn, "Rnl"],
)
assert np.allclose(
    df_tibaitata_et0.loc[filas_completas_rn, "Rn"] + df_tibaitata_et0.loc[filas_completas_rn, "Rnl"],
    df_tibaitata_et0.loc[filas_completas_rn, "Rns"],
)

# Ejemplo numérico de referencia y propagación de NaN con datos sintéticos.
ejemplo_rn = pd.DataFrame({"Rns": [10.0, 5.0, np.nan, 8.0], "Rnl": [3.0, 6.5, 2.0, np.nan]})
resultado_ejemplo_rn = ejemplo_rn["Rns"] - ejemplo_rn["Rnl"]
assert np.allclose(resultado_ejemplo_rn.iloc[:2], [7.0, -1.5])
assert resultado_ejemplo_rn.iloc[2:].isna().all()

n_rn_negativa = int((rn_valida < 0).sum())

print("Primeras filas:")
display(df_tibaitata_et0[["Fecha", "Rns", "Rnl", "Rn"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[["Rns", "Rnl", "Rn"]].isna().sum())
print("Resumen de Rn (MJ/m²/día):")
display(rn_valida.describe())
print(f"Registros con Rn < 0 (se conservan sin cambios): {n_rn_negativa} de {len(rn_valida)}")
if n_rn_negativa:
    print("Ejemplos con Rn negativa (los más negativos):")
    display(
        df_tibaitata_et0.loc[df_tibaitata_et0["Rn"] < 0, ["Fecha", "Rs", "Rs_Rso", "Rns", "Rnl", "Rn"]]
        .nsmallest(5, "Rn")
    )
print("Verificaciones: columnas, fechas y registros originales conservados; Rns y Rnl sin cambios; NaN propagados; Rn = Rns - Rnl.")

Primeras filas:


,Fecha,Rns,Rnl,Rn
0,2005-02-02,4.703,-0.332,5.035
1,2005-02-03,12.420,2.422,9.998
2,2005-02-04,6.551,0.311,6.240
3,2005-02-05,8.696,1.082,7.613
4,2005-02-06,5.524,-0.058,5.582


Valores nulos:


Rns    0
Rnl    0
Rn     0
dtype: int64

Resumen de Rn (MJ/m²/día):


count    2909.000
mean        9.456
std         2.204
min         2.474
25%         7.821
50%         9.278
75%        11.008
max        15.362
Name: Rn, dtype: float64

Registros con Rn < 0 (se conservan sin cambios): 0 de 2909
Verificaciones: columnas, fechas y registros originales conservados; Rns y Rnl sin cambios; NaN propagados; Rn = Rns - Rnl.


## Flujo de calor del suelo

Para el cálculo diario de ET₀ se adopta la aproximación FAO-56 de flujo de calor del suelo despreciable: en periodos de un día, `G ≈ 0` MJ/m²/día. Se asigna `G = 0.0` a todas las fechas. No se modifica ninguna columna existente, no se eliminan registros ni se imputan datos meteorológicos. No se calcula ET₀ en este paso.

In [19]:
assert "G" not in df_tibaitata_et0.columns, "La columna G ya existe; no se sobrescribirá."

columnas_antes_g = df_tibaitata_et0.columns.tolist()
valores_antes_g = df_tibaitata_et0[columnas_antes_g].copy()
fechas_antes_g = df_tibaitata_et0["Fecha"].copy()
numero_registros_antes_g = len(df_tibaitata_et0)
forma_limpio_antes_g = df_tibaitata_limpio.shape

# Aproximación diaria FAO-56: flujo de calor del suelo despreciable (MJ/m²/día).
G_DIARIO = 0.0
df_tibaitata_et0["G"] = G_DIARIO

# Datos originales intactos; sin eliminar fechas ni registros.
assert len(df_tibaitata_et0) == numero_registros_antes_g
assert df_tibaitata_et0[columnas_antes_g].equals(valores_antes_g)
assert df_tibaitata_et0.columns[:len(columnas_antes_g)].tolist() == columnas_antes_g
assert df_tibaitata_et0.columns.tolist() == columnas_antes_g + ["G"]
assert df_tibaitata_et0["Fecha"].equals(fechas_antes_g)
assert df_tibaitata_limpio.shape == forma_limpio_antes_g

# G tiene una fila por registro, sin nulos y todos sus valores son cero.
assert len(df_tibaitata_et0["G"]) == len(df_tibaitata_et0)
assert df_tibaitata_et0["G"].notna().all()
assert (df_tibaitata_et0["G"] == 0).all()
assert df_tibaitata_et0["G"].nunique() == 1

print("G = 0 MJ/m²/día (aproximación diaria FAO-56: flujo de calor del suelo despreciable).")
print(f"Filas en G: {len(df_tibaitata_et0['G'])} de {len(df_tibaitata_et0)} | valores distintos de cero: {int((df_tibaitata_et0['G'] != 0).sum())}")
print("Verificaciones: columnas y registros originales conservados; sin nulos; G = 0 en todas las fechas.")
display(df_tibaitata_et0[["Fecha", "G"]].head())
print("Valores nulos:", df_tibaitata_et0["G"].isna().sum())

G = 0 MJ/m²/día (aproximación diaria FAO-56: flujo de calor del suelo despreciable).
Filas en G: 2909 de 2909 | valores distintos de cero: 0
Verificaciones: columnas y registros originales conservados; sin nulos; G = 0 en todas las fechas.


,Fecha,G
0,2005-02-02,0.0
1,2005-02-03,0.0
2,2005-02-04,0.0
3,2005-02-05,0.0
4,2005-02-06,0.0


Valores nulos: 0


## Evapotranspiración de referencia (ET₀)

Se calcula `ET0` (mm/día) con la ecuación FAO-56 Penman-Monteith diaria:

`ET0 = (0.408 * delta * (Rn - G) + gamma * 900 / (T_mean + 273) * u2 * (es - ea)) / (delta + gamma * (1 + 0.34 * u2))`

- Entradas: `T_mean` (°C), `delta` (kPa/°C), `Rn` y `G` (MJ/m²/día), `gamma` (kPa/°C), `Wind_speed` (m/s) y `es`, `ea` (kPa).
- `Wind_speed` ya está medida a 2 m y se usa directamente como `u2`, sin conversión.
- Si falta o no es finita alguna entrada, `ET0` queda en NaN. También queda en NaN si `T_mean + 273 <= 0` o si el denominador no es finito y positivo, para evitar divisiones por cero e infinitos.
- No se eliminan fechas, no se imputan datos, no se modifican los cálculos intermedios y no se fuerzan a cero los valores negativos.
- Los valores negativos o físicamente sospechosos solo se reportan. No se realiza la auditoría general ni se exporta el archivo.

In [20]:
columnas_et0_requeridas = ["T_mean", "delta", "Rn", "G", "gamma", "Wind_speed", "es", "ea"]
assert set(columnas_et0_requeridas).issubset(df_tibaitata_et0.columns)
assert "ET0" not in df_tibaitata_et0.columns, "La columna ET0 ya existe; no se sobrescribirá."

def calcular_et0(df):
    """FAO-56 Penman-Monteith diaria (mm/día). Devuelve una Serie sin modificar `df`.

    Wind_speed se usa directamente como u2 (ya está a 2 m).
    Queda NaN si falta o no es finita alguna entrada, si T_mean + 273 <= 0
    o si el denominador no es finito y positivo (evita divisiones por cero).
    """
    t, d, rn, g, gam, u2, es_, ea_ = (df[c].astype(float) for c in columnas_et0_requeridas)

    entradas_validas = np.isfinite(df[columnas_et0_requeridas].astype(float)).all(axis=1)
    temperatura_valida = (t + 273) > 0
    denominador = d + gam * (1 + 0.34 * u2)
    denominador_valido = np.isfinite(denominador) & (denominador > 0)
    calculable = entradas_validas & temperatura_valida & denominador_valido

    t_c, d_c, rn_c, g_c = t.where(calculable), d.where(calculable), rn.where(calculable), g.where(calculable)
    gam_c, u2_c, es_c, ea_c = gam.where(calculable), u2.where(calculable), es_.where(calculable), ea_.where(calculable)

    numerador = (
        0.408 * d_c * (rn_c - g_c)
        + gam_c * (900 / (t_c + 273)) * u2_c * (es_c - ea_c)
    )
    et0 = numerador / (d_c + gam_c * (1 + 0.34 * u2_c))
    return et0.where(np.isfinite(et0))

# --- Verificación con ejemplos independientes ---
def et0_escalar(t, d, rn, g, gam, u2, es_, ea_):
    return (
        0.408 * d * (rn - g) + gam * 900 / (t + 273) * u2 * (es_ - ea_)
    ) / (d + gam * (1 + 0.34 * u2))

# Primera fila: Ejemplo 18 de FAO-56 (Bruselas, 6 de mayo; ET0 publicada ≈ 3.9 mm/día).
# Las demás filas son casos sintéticos: normal, Rn negativa y entradas no válidas.
ejemplos_et0 = pd.DataFrame(
    [
        [16.9, 0.122, 13.28, 0.0, 0.0666, 2.078, 1.997, 1.409],
        [12.0, 0.090, 2.0, 0.0, 0.05, 1.0, 1.4, 1.2],
        [12.0, 0.090, -2.0, 0.0, 0.05, 1.0, 1.4, 1.2],
        [np.nan, 0.090, 2.0, 0.0, 0.05, 1.0, 1.4, 1.2],
        [12.0, np.nan, 2.0, 0.0, 0.05, 1.0, 1.4, 1.2],
        [12.0, 0.090, np.nan, 0.0, 0.05, 1.0, 1.4, 1.2],
        [12.0, 0.090, 2.0, 0.0, 0.05, np.inf, 1.4, 1.2],
        [-273.0, 0.090, 2.0, 0.0, 0.05, 1.0, 1.4, 1.2],
        [12.0, 0.0, 2.0, 0.0, 0.0, 1.0, 1.4, 1.2],
    ],
    columns=columnas_et0_requeridas,
    index=["FAO-56 ej. 18", "normal", "Rn negativa", "T_mean NaN", "delta NaN",
           "Rn NaN", "Wind_speed inf", "T_mean + 273 = 0", "denominador = 0"],
)

resultado_ejemplos_et0 = calcular_et0(ejemplos_et0)
for caso in ["FAO-56 ej. 18", "normal", "Rn negativa"]:
    esperado = et0_escalar(*ejemplos_et0.loc[caso, columnas_et0_requeridas].tolist())
    assert math.isclose(resultado_ejemplos_et0[caso], esperado, rel_tol=1e-12), caso
assert abs(resultado_ejemplos_et0["FAO-56 ej. 18"] - 3.9) < 0.1
assert resultado_ejemplos_et0["Rn negativa"] < 0, "Un resultado negativo no debe forzarse a cero."
for caso in ["T_mean NaN", "delta NaN", "Rn NaN", "Wind_speed inf", "T_mean + 273 = 0", "denominador = 0"]:
    assert np.isnan(resultado_ejemplos_et0[caso]), f"{caso} debe producir NaN."
print("Ejemplos de verificación (ET0 en mm/día):")
display(pd.concat([ejemplos_et0, resultado_ejemplos_et0.rename("ET0")], axis=1))
print("Ejemplos verificados contra un cálculo escalar independiente.")

# --- Cálculo sobre df_tibaitata_et0 ---
columnas_antes_et0 = df_tibaitata_et0.columns.tolist()
valores_antes_et0 = df_tibaitata_et0[columnas_antes_et0].copy()
numero_registros_antes_et0 = len(df_tibaitata_et0)
forma_limpio_antes_et0 = df_tibaitata_limpio.shape

df_tibaitata_et0["ET0"] = calcular_et0(df_tibaitata_et0)

# Datos originales y cálculos intermedios intactos; sin eliminar fechas.
assert len(df_tibaitata_et0) == numero_registros_antes_et0
assert df_tibaitata_et0[columnas_antes_et0].equals(valores_antes_et0)
assert df_tibaitata_et0.columns.tolist() == columnas_antes_et0 + ["ET0"]
assert df_tibaitata_limpio.shape == forma_limpio_antes_et0

et0 = df_tibaitata_et0["ET0"]
et0_valida = et0.dropna()
filas_completas_et0 = df_tibaitata_et0[columnas_et0_requeridas].notna().all(axis=1)

# NaN propagados (sin imputación) y valores finitos.
assert et0.loc[~filas_completas_et0].isna().all(), "ET0 no debe tener valor si falta una entrada."
assert np.isfinite(et0_valida).all(), "ET0 contiene valores infinitos."

# Recálculo independiente sobre las filas con ET0.
fv = df_tibaitata_et0.loc[et0.notna()]
et0_control = (
    0.408 * fv["delta"] * (fv["Rn"] - fv["G"])
    + fv["gamma"] * 900 / (fv["T_mean"] + 273) * fv["Wind_speed"] * (fv["es"] - fv["ea"])
) / (fv["delta"] + fv["gamma"] * (1 + 0.34 * fv["Wind_speed"]))
assert np.allclose(fv["ET0"], et0_control)

# --- Conteos ---
n_total_et0 = len(et0)
n_validas_et0 = int(et0.notna().sum())
n_nulas_et0 = int(et0.isna().sum())
n_negativas_et0 = int((et0 < 0).sum())
n_nulas_con_entradas_completas = int((et0.isna() & filas_completas_et0).sum())

# --- Valores negativos o físicamente sospechosos (solo se reportan) ---
ET0_MAX_SOSPECHOSO = 10.0  # mm/día; valores diarios superiores son excepcionales en cualquier clima
criterios_sospechosos = pd.DataFrame(
    {
        "ET0 < 0": et0 < 0,
        f"ET0 > {ET0_MAX_SOSPECHOSO:g} mm/día": et0 > ET0_MAX_SOSPECHOSO,
        "Wind_speed < 0": df_tibaitata_et0["Wind_speed"] < 0,
        "es < ea (HR > 100 %)": df_tibaitata_et0["es"] < df_tibaitata_et0["ea"],
        "Rn - G < 0": (df_tibaitata_et0["Rn"] - df_tibaitata_et0["G"]) < 0,
        "delta <= 0": df_tibaitata_et0["delta"] <= 0,
        "gamma <= 0": df_tibaitata_et0["gamma"] <= 0,
    }
)
resumen_sospechosos = criterios_sospechosos.sum().rename("registros").to_frame()

print(f"Registros: {n_total_et0}")
print(f"ET0 válidas: {n_validas_et0}")
print(f"ET0 nulas: {n_nulas_et0} (con las 8 entradas completas pero no calculables: {n_nulas_con_entradas_completas})")
print(f"ET0 negativas: {n_negativas_et0}")
print("Primeras filas con variables de entrada y ET0:")
display(df_tibaitata_et0[["Fecha"] + columnas_et0_requeridas + ["ET0"]].head())
print("Valores nulos por variable de entrada y ET0:")
display(df_tibaitata_et0[columnas_et0_requeridas + ["ET0"]].isna().sum())
print("Estadísticas descriptivas de ET0 (mm/día):")
display(et0_valida.describe())
print("Registros negativos o físicamente sospechosos (solo informativo; no se corrige nada):")
display(resumen_sospechosos)
sospechosos_et0 = df_tibaitata_et0.loc[criterios_sospechosos.any(axis=1), ["Fecha"] + columnas_et0_requeridas + ["ET0"]]
if sospechosos_et0.empty:
    print("No se encontraron registros negativos ni sospechosos.")
else:
    print(f"Registros afectados: {len(sospechosos_et0)} (se conservan sin cambios). Primeros casos:")
    display(sospechosos_et0.head(10))
print("Verificaciones: columnas, fechas y registros originales conservados; cálculos intermedios sin cambios; NaN propagados; ET0 finita y coincide con un recálculo independiente.")

Ejemplos de verificación (ET0 en mm/día):


,T_mean,delta,Rn,G,gamma,Wind_speed,es,ea,ET0
FAO-56 ej. 18,16.9,0.122,13.28,0.0,0.067,2.078,1.997,1.409,3.877
normal,12.0,0.090,2.00,0.0,0.050,1.000,1.400,1.200,0.669
Rn negativa,12.0,0.090,-2.00,0.0,0.050,1.000,1.400,1.200,-0.267
T_mean NaN,NaN,0.090,2.00,0.0,0.050,1.000,1.400,1.200,NaN
delta NaN,12.0,NaN,2.00,0.0,0.050,1.000,1.400,1.200,NaN
Rn NaN,12.0,0.090,NaN,0.0,0.050,1.000,1.400,1.200,NaN
Wind_speed inf,12.0,0.090,2.00,0.0,0.050,inf,1.400,1.200,NaN
T_mean + 273 = 0,-273.0,0.090,2.00,0.0,0.050,1.000,1.400,1.200,NaN
denominador = 0,12.0,0.000,2.00,0.0,0.000,1.000,1.400,1.200,NaN


Ejemplos verificados contra un cálculo escalar independiente.


Registros: 2909
ET0 válidas: 2909
ET0 nulas: 0 (con las 8 entradas completas pero no calculables: 0)
ET0 negativas: 0
Primeras filas con variables de entrada y ET0:


,Fecha,T_mean,delta,Rn,G,gamma,Wind_speed,es,ea,ET0
0,2005-02-02,12.2,0.094,5.035,0.0,0.05,1.645,1.579,1.414,1.373
1,2005-02-03,12.9,0.097,9.998,0.0,0.05,1.965,1.638,1.376,2.653
2,2005-02-04,12.7,0.096,6.240,0.0,0.05,2.149,1.566,1.397,1.656
3,2005-02-05,14.7,0.108,7.613,0.0,0.05,2.558,1.772,1.428,2.352
4,2005-02-06,14.8,0.109,5.582,0.0,0.05,1.965,1.733,1.551,1.582


Valores nulos por variable de entrada y ET0:


... (2 outputs left out)

count    2909.000
mean        2.652
std         0.592
min         0.907
25%         2.234
50%         2.619
75%         3.074
max         4.449
Name: ET0, dtype: float64

Registros negativos o físicamente sospechosos (solo informativo; no se corrige nada):


,registros
ET0 < 0,0
ET0 > 10 mm/día,0
Wind_speed < 0,0
es < ea (HR > 100 %),0
Rn - G < 0,0
delta <= 0,0
gamma <= 0,0


No se encontraron registros negativos ni sospechosos.
Verificaciones: columnas, fechas y registros originales conservados; cálculos intermedios sin cambios; NaN propagados; ET0 finita y coincide con un recálculo independiente.


## Auditoría final del dataset maestro

Se audita `df_tibaitata_et0` antes de exportarlo. La auditoría es de solo lectura: no imputa, no corrige, no elimina registros y no sustituye valores negativos de ET₀. Verifica fechas (comparando contra `df_tibaitata_limpio`), duplicados, variables originales, columnas intermedias, unidades, infinitos, nulos, uso directo de `Wind_speed` como `u2` y ET₀ negativa. Los datos de la estación (altitud, latitud, longitud) se leen de `df_estacion` y se contrastan con los valores derivados del dataset. Un error estructural bloquea la exportación; las advertencias solo se reportan.

In [21]:
# ============================================================
# Auditoría final de df_tibaitata_et0 (solo lectura: no corrige, no imputa, no elimina)
# ============================================================
import math

snapshot_et0_auditoria = df_tibaitata_et0.copy(deep=True)
snapshot_limpio_auditoria = df_tibaitata_limpio.copy(deep=True)

resultados_auditoria = []
errores_estructurales = []


def registrar(n, verificacion, ok, detalle, estructural=True):
    """Registra el resultado de una verificación. Un fallo estructural bloquea la exportación."""
    estado = "OK" if ok else ("ERROR" if estructural else "ADVERTENCIA")
    resultados_auditoria.append((n, verificacion, estado, detalle))
    if not ok and estructural:
        errores_estructurales.append(f"[{n}] {verificacion}: {detalle}")


def cercano(a, b, rtol=1e-9, atol=1e-12):
    """Igualdad numérica con tolerancia; NaN en la misma posición cuenta como igual."""
    return bool(np.allclose(np.asarray(a, dtype=float), np.asarray(b, dtype=float),
                            rtol=rtol, atol=atol, equal_nan=True))


def cercano_escalar(serie, valor, rtol=1e-9, atol=1e-12):
    s = serie.dropna()
    return len(s) > 0 and bool(np.allclose(s.to_numpy(float), valor, rtol=rtol, atol=atol))


def et0_penman_monteith(df, u2):
    """Recálculo independiente de FAO-56 Penman-Monteith diaria (mm/día)."""
    with np.errstate(all="ignore"):
        numerador = (
            0.408 * df["delta"] * (df["Rn"] - df["G"])
            + df["gamma"] * 900 / (df["T_mean"] + 273) * u2 * (df["es"] - df["ea"])
        )
        return numerador / (df["delta"] + df["gamma"] * (1 + 0.34 * u2))


ORIGINALES = list(parametros_et0.values())  # RH_mean, Radiation_Wh_m2, Tmax, Tmin, Wind_speed
INTERMEDIAS = [
    "T_mean", "es_tmax", "es_tmin", "es", "es_tmean", "delta", "ea", "P", "gamma", "Rs",
    "J", "phi_rad", "dr", "solar_declination", "omega_s", "Ra", "Rso", "Rns",
    "Tmax_K", "Tmin_K", "ea_sqrt", "Rs_Rso_sin_limitar", "Rs_Rso", "Rnl", "Rn", "G",
]
COLUMNAS_ESPERADAS = ["Fecha"] + ORIGINALES + INTERMEDIAS + ["ET0"]
df = df_tibaitata_et0
n_filas = len(df)

# ------------------------------------------------------------
print("=== 0. Datos de la estación (df_estacion) ===")
assert len(df_estacion) == 1, "df_estacion debe contener una sola estación."
display(df_estacion[["codigo", "nombre", "municipio", "departamento", "altitud", "longitud", "latitud"]])
alt_est = float(pd.to_numeric(df_estacion["altitud"]).iloc[0])
lat_est = float(pd.to_numeric(df_estacion["latitud"]).iloc[0])
lon_est = float(pd.to_numeric(df_estacion["longitud"]).iloc[0])
p_esperada = 101.3 * ((293 - 0.0065 * alt_est) / 293) ** 5.26
coef_rso_esperado = 0.75 + 2e-5 * alt_est
print(f"Altitud (z)  = {alt_est:g} m   -> P, gamma, Rso")
print(f"Latitud      = {lat_est} °  -> phi_rad, dr, delta solar, omega_s, Ra")
print(f"Longitud     = {lon_est} °  (no interviene en el cálculo diario de ET0)")
print(f"Valores derivados esperados: P = {p_esperada:.6f} kPa | gamma = {0.000665 * p_esperada:.6f} kPa/°C | "
      f"phi = {np.deg2rad(lat_est):.6f} rad | Rso/Ra = {coef_rso_esperado:.5f}")
fallos_estacion = []
if not (alt_est == float(ALTITUD_TIBAITATA_M) and lat_est == float(LATITUD_TIBAITATA_DEG)):
    fallos_estacion.append("las variables del notebook no coinciden con df_estacion")
if not cercano_escalar(df["P"], p_esperada):
    fallos_estacion.append("P no corresponde a la altitud de df_estacion")
if not cercano_escalar(df["gamma"], 0.000665 * p_esperada):
    fallos_estacion.append("gamma no corresponde a la altitud de df_estacion")
if not cercano_escalar(df["phi_rad"], np.deg2rad(lat_est)):
    fallos_estacion.append("phi_rad no corresponde a la latitud de df_estacion")
if not cercano_escalar(df["Rso"] / df["Ra"], coef_rso_esperado):
    fallos_estacion.append("Rso/Ra no corresponde a la altitud de df_estacion")
registrar(0, "Datos de la estación tomados de df_estacion (altitud, latitud)", not fallos_estacion,
          "; ".join(fallos_estacion) or f"z = {alt_est:g} m, latitud = {lat_est}°; P, gamma, phi_rad y Rso/Ra coinciden")

# ------------------------------------------------------------
print("\n=== 1. Fechas: df_tibaitata_limpio (largo) vs df_tibaitata_et0 (ancho) ===")
fechas_limpio = pd.DatetimeIndex(df_tibaitata_limpio["Fecha"].unique()).sort_values()
fechas_et0 = pd.DatetimeIndex(df["Fecha"])
faltantes_en_et0 = fechas_limpio.difference(fechas_et0)
sobrantes_en_et0 = fechas_et0.difference(fechas_limpio)
print(f"Fechas únicas en limpio: {len(fechas_limpio)} | filas en et0: {n_filas}")
print(f"Fechas de limpio ausentes en et0: {len(faltantes_en_et0)} | fechas de et0 inexistentes en limpio: {len(sobrantes_en_et0)}")
rango_completo = pd.date_range(fechas_et0.min(), fechas_et0.max(), freq="D")
dias_sin_registro = rango_completo.difference(fechas_et0)
print(f"Rango: {fechas_et0.min().date()} a {fechas_et0.max().date()} | días calendario: {len(rango_completo)} | "
      f"días sin registro en el rango (informativo): {len(dias_sin_registro)}")
registrar(1, "Todas las fechas originales están representadas",
          len(faltantes_en_et0) == 0 and len(sobrantes_en_et0) == 0 and df_tibaitata_limpio["Fecha"].notna().all(),
          f"{len(fechas_limpio)} fechas en limpio; {len(faltantes_en_et0)} ausentes y {len(sobrantes_en_et0)} sobrantes en et0")

# ------------------------------------------------------------
print("\n=== 2. Fechas duplicadas ===")
n_duplicadas = int(df["Fecha"].duplicated().sum())
print(f"Fechas duplicadas: {n_duplicadas} | fechas nulas: {int(df['Fecha'].isna().sum())} | "
      f"orden cronológico: {df['Fecha'].is_monotonic_increasing}")
registrar(2, "No existen fechas duplicadas", n_duplicadas == 0 and df["Fecha"].notna().all(),
          f"{n_duplicadas} fechas duplicadas")
registrar("2b", "Fechas en orden cronológico creciente", df["Fecha"].is_monotonic_increasing,
          "orden creciente" if df["Fecha"].is_monotonic_increasing else "las fechas no están ordenadas", estructural=False)

# ------------------------------------------------------------
print("\n=== 3. Variables meteorológicas originales ===")
faltan_originales = [c for c in ORIGINALES if c not in df.columns]
referencia_pivot = (
    df_tibaitata_limpio[df_tibaitata_limpio["parametro_limpio"].isin(parametros_et0)]
    .assign(parametro_limpio=lambda d: d["parametro_limpio"].map(parametros_et0))
    .pivot(index="Fecha", columns="parametro_limpio", values="Valor")
    .reindex(index=fechas_et0, columns=ORIGINALES)
)
actuales = df.set_index("Fecha")[[c for c in ORIGINALES if c in df.columns]]
valores_iguales = (
    not faltan_originales
    and bool(((actuales == referencia_pivot) | (actuales.isna() & referencia_pivot.isna())).all().all())
)
print(f"Columnas originales: {ORIGINALES}")
print(f"Faltantes: {faltan_originales or 'ninguna'} | valores idénticos a un nuevo pivotado de df_tibaitata_limpio: {valores_iguales}")
registrar(3, "Las cinco variables meteorológicas originales siguen presentes y sin alterar",
          not faltan_originales and valores_iguales,
          f"faltantes: {faltan_originales or 'ninguna'}; valores idénticos al pivotado independiente: {valores_iguales}")

# ------------------------------------------------------------
print("\n=== 4. Columnas intermedias y ET0 ===")
faltan_cols = [c for c in INTERMEDIAS + ["ET0"] if c not in df.columns]
extras = [c for c in df.columns if c not in COLUMNAS_ESPERADAS]
duplicadas_cols = df.columns[df.columns.duplicated()].tolist()
mismo_orden = df.columns.tolist() == COLUMNAS_ESPERADAS
print(f"Columnas en el DataFrame: {df.shape[1]} | esperadas: {len(COLUMNAS_ESPERADAS)}")
print(f"Faltantes: {faltan_cols or 'ninguna'} | no previstas: {extras or 'ninguna'} | nombres repetidos: {duplicadas_cols or 'ninguno'} | orden esperado: {mismo_orden}")
registrar(4, "Todas las columnas intermedias y ET0 están presentes",
          not faltan_cols and not duplicadas_cols, f"faltantes: {faltan_cols or 'ninguna'}; repetidas: {duplicadas_cols or 'ninguna'}")
registrar("4b", "No hay columnas no previstas y el orden es el esperado", not extras and mismo_orden,
          f"no previstas: {extras or 'ninguna'}; orden esperado: {mismo_orden}", estructural=False)

# ------------------------------------------------------------
print("\n=== 5. Coherencia de unidades ===")
pi = math.pi
rangos_unidades = [
    # (columna, unidad, mínimo plausible, máximo plausible)
    ("RH_mean", "%", 0, 100),
    ("Radiation_Wh_m2", "Wh/m²/día", 0, 12000),
    ("Tmax", "°C", -10, 45),
    ("Tmin", "°C", -15, 35),
    ("Wind_speed", "m/s", 0, 20),
    ("T_mean", "°C", -10, 40),
    ("es_tmax", "kPa", 0, 10), ("es_tmin", "kPa", 0, 10), ("es", "kPa", 0, 10), ("es_tmean", "kPa", 0, 10),
    ("delta", "kPa/°C", 0, 0.35),
    ("ea", "kPa", 0, 10),
    ("P", "kPa", 50, 101.4),
    ("gamma", "kPa/°C", 0, 0.07),
    ("Rs", "MJ/m²/día", 0, 40),
    ("J", "día del año", 1, 366),
    ("phi_rad", "rad", -pi / 2, pi / 2),
    ("dr", "adimensional", 0.967, 1.033),
    ("solar_declination", "rad", -0.409 - 1e-9, 0.409 + 1e-9),
    ("omega_s", "rad", 0, pi),
    ("Ra", "MJ/m²/día", 0, 45),
    ("Rso", "MJ/m²/día", 0, 45),
    ("Rns", "MJ/m²/día", 0, 40),
    ("Tmax_K", "K", 230, 330), ("Tmin_K", "K", 230, 330),
    ("ea_sqrt", "kPa^0.5", 0, 3.2),
    ("Rs_Rso_sin_limitar", "adimensional", 0, 2),
    ("Rs_Rso", "adimensional", 0, 1),
    ("Rnl", "MJ/m²/día", -5, 15),
    ("Rn", "MJ/m²/día", -5, 30),
    ("G", "MJ/m²/día", 0, 0),
    ("ET0", "mm/día", -5, 15),
]
filas_unidades = []
for columna, unidad, minimo, maximo in rangos_unidades:
    serie = df[columna].dropna()
    fuera = int(((serie < minimo) | (serie > maximo)).sum())
    filas_unidades.append({
        "variable": columna, "unidad": unidad, "mín. observado": serie.min(), "máx. observado": serie.max(),
        "rango plausible": f"[{minimo:.4g}, {maximo:.4g}]", "fuera de rango": fuera,
    })
tabla_unidades = pd.DataFrame(filas_unidades).set_index("variable")
display(tabla_unidades)
total_fuera = int(tabla_unidades["fuera de rango"].sum())
registrar(5, "Valores de cada variable dentro de rangos plausibles para su unidad", total_fuera == 0,
          f"{total_fuera} valores fuera de rango en {int((tabla_unidades['fuera de rango'] > 0).sum())} variables")

identidades = {
    "Rs = Radiation_Wh_m2 * 0.0036 (Wh -> MJ)": cercano(df["Rs"], df["Radiation_Wh_m2"] * 0.0036),
    "T_mean = (Tmax + Tmin) / 2": cercano(df["T_mean"], (df["Tmax"] + df["Tmin"]) / 2),
    "Tmax_K = Tmax + 273.16 y Tmin_K = Tmin + 273.16": cercano(df["Tmax_K"], df["Tmax"] + 273.16) and cercano(df["Tmin_K"], df["Tmin"] + 273.16),
    "ea = es * RH_mean / 100 (RH en %)": cercano(df["ea"], df["es"] * df["RH_mean"] / 100),
    "gamma = 0.000665 * P": cercano(df["gamma"], 0.000665 * df["P"]),
    "J = día del año de Fecha": bool((df["J"] == df["Fecha"].dt.dayofyear).all()),
    "Rns = 0.77 * Rs": cercano(df["Rns"], 0.77 * df["Rs"]),
    "Rn = Rns - Rnl": cercano(df["Rn"], df["Rns"] - df["Rnl"]),
    "Rs_Rso = min(Rs_Rso_sin_limitar, 1)": cercano(df["Rs_Rso"], df["Rs_Rso_sin_limitar"].clip(upper=1)),
}
for descripcion, ok in identidades.items():
    print(f"  [{'OK' if ok else 'ERROR'}] {descripcion}")
registrar("5b", "Relaciones entre variables y conversiones de unidades correctas", all(identidades.values()),
          "todas las identidades se cumplen" if all(identidades.values())
          else "fallan: " + "; ".join(k for k, v in identidades.items() if not v))
n_ea_mayor_es = int((df["ea"] > df["es"]).sum())
n_tmin_mayor_tmax = int((df["Tmin"] > df["Tmax"]).sum())
n_rs_mayor_ra = int((df["Rs"] > df["Ra"]).sum())
print(f"  Informativo: ea > es: {n_ea_mayor_es} | Tmin > Tmax: {n_tmin_mayor_tmax} | Rs > Ra: {n_rs_mayor_ra}")
registrar("5c", "Coherencia física: ea <= es, Tmin <= Tmax, Rs <= Ra",
          n_ea_mayor_es == 0 and n_tmin_mayor_tmax == 0 and n_rs_mayor_ra == 0,
          f"ea > es: {n_ea_mayor_es}; Tmin > Tmax: {n_tmin_mayor_tmax}; Rs > Ra: {n_rs_mayor_ra}", estructural=False)

# ------------------------------------------------------------
print("\n=== 6. Valores infinitos ===")
columnas_numericas = df.select_dtypes(include="number").columns
infinitos_por_columna = np.isinf(df[columnas_numericas]).sum()
n_infinitos = int(infinitos_por_columna.sum())
print(f"Columnas numéricas revisadas: {len(columnas_numericas)} | valores infinitos: {n_infinitos}")
if n_infinitos:
    display(infinitos_por_columna[infinitos_por_columna > 0])
registrar(6, "No existen valores infinitos", n_infinitos == 0, f"{n_infinitos} valores infinitos")

# ------------------------------------------------------------
print("\n=== 7. Valores nulos: conservación y conteo ===")
nulos_por_columna = df.isna().sum()
validos_por_parametro = (
    df_tibaitata_limpio[df_tibaitata_limpio["parametro_limpio"].isin(parametros_et0)]
    .dropna(subset=["Valor"])
    .groupby("parametro_limpio")["Valor"].size()
)
nulos_esperados = {
    columna: len(fechas_limpio) - int(validos_por_parametro.get(nombre, 0))
    for nombre, columna in parametros_et0.items()
}
originales_coinciden = all(int(nulos_por_columna[c]) == nulos_esperados[c] for c in ORIGINALES)
print("Nulos de las variables originales: observados vs esperados desde df_tibaitata_limpio (largo)")
display(pd.DataFrame({
    "nulos en et0": nulos_por_columna[ORIGINALES],
    "esperados desde limpio": pd.Series(nulos_esperados),
}))
registrar(7, "Nulos de las variables originales coinciden con los de df_tibaitata_limpio", originales_coinciden,
          "; ".join(f"{c}: {int(nulos_por_columna[c])} vs {nulos_esperados[c]}" for c in ORIGINALES))

entradas_por_columna = {
    "T_mean": ["Tmax", "Tmin"], "es_tmax": ["Tmax"], "es_tmin": ["Tmin"], "es": ["Tmax", "Tmin"],
    "es_tmean": ["T_mean"], "delta": ["T_mean"], "ea": ["es", "RH_mean"], "P": [], "gamma": ["P"],
    "Rs": ["Radiation_Wh_m2"], "J": [], "phi_rad": [], "dr": [], "solar_declination": [], "omega_s": [],
    "Ra": [], "Rso": ["Ra"], "Rns": ["Rs"], "Tmax_K": ["Tmax"], "Tmin_K": ["Tmin"], "ea_sqrt": ["ea"],
    "Rs_Rso_sin_limitar": ["Rs", "Rso"], "Rs_Rso": ["Rs", "Rso"],
    "Rnl": ["Tmax_K", "Tmin_K", "ea_sqrt", "Rs_Rso"], "Rn": ["Rns", "Rnl"], "G": [],
    "ET0": ["T_mean", "delta", "Rn", "G", "gamma", "Wind_speed", "es", "ea"],
}
valor_donde_falta_entrada = {}
nulo_sin_entrada_faltante = {}
for columna, entradas in entradas_por_columna.items():
    falta_entrada = df[entradas].isna().any(axis=1) if entradas else pd.Series(False, index=df.index)
    valor_donde_falta_entrada[columna] = int((falta_entrada & df[columna].notna()).sum())
    nulo_sin_entrada_faltante[columna] = int((~falta_entrada & df[columna].isna()).sum())
mal_propagados = {c: n for c, n in valor_donde_falta_entrada.items() if n}
nulos_extra = {c: n for c, n in nulo_sin_entrada_faltante.items() if n}
registrar("7b", "Los nulos se propagan: ninguna columna derivada tiene valor cuando falta una entrada",
          not mal_propagados, f"columnas con valor pese a entrada faltante: {mal_propagados or 'ninguna'}")
registrar("7c", "Ninguna columna derivada tiene nulos adicionales a los de sus entradas", not nulos_extra,
          f"nulos adicionales: {nulos_extra or 'ninguno'}", estructural=False)
registrar("7d", "El conteo de nulos por columna es consistente",
          int(df.isna().sum().sum()) == int(nulos_por_columna.sum()) and len(nulos_por_columna) == df.shape[1],
          f"total de nulos en el DataFrame: {int(nulos_por_columna.sum())}")
print(f"Total de nulos en el DataFrame: {int(nulos_por_columna.sum())} | columnas con nulos: {int((nulos_por_columna > 0).sum())}")
filas_sin_ninguna_original = int(df[ORIGINALES].isna().all(axis=1).sum())
print(f"Filas con las cinco variables originales ausentes (se conservan): {filas_sin_ninguna_original}")

# ------------------------------------------------------------
print("\n=== 8. Registros conservados ===")
df_limpio_en_disco = pd.read_pickle(DATASET_LIMPIO_PATH)
limpio_sin_cambios = df_tibaitata_limpio.equals(df_limpio_en_disco) and df_tibaitata_limpio.equals(snapshot_limpio_auditoria)
print(f"Filas en et0: {n_filas} | fechas únicas en limpio: {len(fechas_limpio)} | fechas únicas en et0: {df['Fecha'].nunique()}")
print(f"df_tibaitata_limpio: {df_tibaitata_limpio.shape} en memoria vs {df_limpio_en_disco.shape} en disco | idéntico: {limpio_sin_cambios}")
registrar(8, "No se eliminaron registros: una fila por cada fecha de df_tibaitata_limpio",
          n_filas == len(fechas_limpio) == df["Fecha"].nunique(),
          f"{n_filas} filas = {len(fechas_limpio)} fechas únicas en limpio")
registrar("8b", "df_tibaitata_limpio permanece intacto (igual al archivo original en disco)", limpio_sin_cambios,
          f"forma {df_tibaitata_limpio.shape}; idéntico al pickle de origen: {limpio_sin_cambios}")

# ------------------------------------------------------------
print("\n=== 9. Wind_speed usada directamente como u2 ===")
et0_con_wind_directo = et0_penman_monteith(df, df["Wind_speed"])
et0_con_wind_convertido = et0_penman_monteith(df, df["Wind_speed"] * 4.87 / np.log(67.8 * 10 - 5.42))
columnas_viento_extra = [c for c in df.columns if c.lower() in {"u2", "u10", "wind_2m", "wind_speed_2m", "u2_calc"}]
coincide_directo = cercano(df["ET0"], et0_con_wind_directo)
diferencia_si_se_convirtiera = float(np.nanmax(np.abs(df["ET0"] - et0_con_wind_convertido)))
print(f"ET0 almacenada = ET0 recalculada con u2 = Wind_speed: {coincide_directo}")
print(f"Contraprueba: si se hubiera convertido el viento de 10 m a 2 m, la diferencia máxima sería {diferencia_si_se_convirtiera:.3f} mm/día")
print(f"Columnas de viento adicionales (u2, u10, ...): {columnas_viento_extra or 'ninguna'}")
registrar(9, "Wind_speed se usó directamente como u2 (sin conversión de altura)",
          coincide_directo and diferencia_si_se_convirtiera > 1e-3 and not columnas_viento_extra,
          f"ET0 coincide con u2 = Wind_speed: {coincide_directo}; diferencia con viento convertido: {diferencia_si_se_convirtiera:.3f} mm/día")

# ------------------------------------------------------------
print("\n=== 10. ET0 en mm/día ===")
et0_valida = df["ET0"].dropna()
recalculo_completo = cercano(df["ET0"], et0_con_wind_directo)
mediana_et0 = float(et0_valida.median())
razon_energia = float((et0_valida / (0.408 * df.loc[et0_valida.index, "Rn"])).median())
print(f"ET0: mín = {et0_valida.min():.3f} | mediana = {mediana_et0:.3f} | media = {et0_valida.mean():.3f} | máx = {et0_valida.max():.3f} mm/día")
print(f"Mediana de ET0 / (0.408 * Rn): {razon_energia:.2f} (0.408 convierte MJ/m²/día a mm/día)")
registrar(10, "ET0 está en mm/día (ecuación FAO-56 con entradas en MJ/m²/día, kPa, °C y m/s)",
          recalculo_completo and 0.5 <= mediana_et0 <= 8 and et0_valida.max() <= 15,
          f"recálculo coincide: {recalculo_completo}; mediana {mediana_et0:.3f} mm/día; máx {et0_valida.max():.3f} mm/día")

# ------------------------------------------------------------
print("\n=== 11. ET0 negativa (solo se reporta) ===")
et0_negativa = df.loc[df["ET0"] < 0, ["Fecha", "T_mean", "Rn", "Wind_speed", "es", "ea", "ET0"]]
n_negativas_recalculo = int((et0_con_wind_directo < 0).sum())
n_ceros = int((df["ET0"] == 0).sum())
print(f"ET0 < 0: {len(et0_negativa)} | negativas en el recálculo independiente: {n_negativas_recalculo} | ET0 == 0: {n_ceros}")
if len(et0_negativa):
    print("Registros con ET0 negativa (se conservan sin cambios):")
    display(et0_negativa.sort_values("ET0").head(20))
else:
    print("No hay registros con ET0 negativa.")
registrar(11, "ET0 negativa identificada y no sustituida por cero",
          len(et0_negativa) == n_negativas_recalculo and (n_ceros == int((et0_con_wind_directo == 0).sum())),
          f"{len(et0_negativa)} negativas (recálculo: {n_negativas_recalculo}); {n_ceros} valores exactamente cero")

# ------------------------------------------------------------
print("\n=== 12. La auditoría no modificó los datos ===")
registrar(12, "df_tibaitata_et0 no cambió durante la auditoría", df_tibaitata_et0.equals(snapshot_et0_auditoria),
          "idéntico a la copia tomada al iniciar")

# ------------------------------------------------------------
tabla_auditoria = pd.DataFrame(resultados_auditoria, columns=["n", "verificación", "estado", "detalle"])
print("\n=== Resultado de la auditoría ===")
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(tabla_auditoria.set_index("n"))
AUDITORIA_APTA_PARA_EXPORTAR = not errores_estructurales
if AUDITORIA_APTA_PARA_EXPORTAR:
    n_adv = int((tabla_auditoria["estado"] == "ADVERTENCIA").sum())
    print(f"Sin errores estructurales ({n_adv} advertencias). Se puede exportar.")
else:
    print("Errores estructurales que impiden exportar:")
    for e in errores_estructurales:
        print("  -", e)

=== 0. Datos de la estación (df_estacion) ===


,codigo,nombre,municipio,departamento,altitud,longitud,latitud
0,21205420,TIBAITATA [21205420],Mosquera,Cundinamarca,2543,-74.206,4.689


Altitud (z)  = 2543 m   -> P, gamma, Rso
Latitud      = 4.688662 °  -> phi_rad, dr, delta solar, omega_s, Ra
Longitud     = -74.205626 °  (no interviene en el cálculo diario de ET0)
Valores derivados esperados: P = 74.637733 kPa | gamma = 0.049634 kPa/°C | phi = 0.081833 rad | Rso/Ra = 0.80086

=== 1. Fechas: df_tibaitata_limpio (largo) vs df_tibaitata_et0 (ancho) ===
Fechas únicas en limpio: 2909 | filas en et0: 2909
Fechas de limpio ausentes en et0: 0 | fechas de et0 inexistentes en limpio: 0
Rango: 2005-02-02 a 2022-10-31 | días calendario: 6481 | días sin registro en el rango (informativo): 3572

=== 2. Fechas duplicadas ===
Fechas duplicadas: 0 | fechas nulas: 0 | orden cronológico: True

=== 3. Variables meteorológicas originales ===


Columnas originales: ['RH_mean', 'Radiation_Wh_m2', 'Tmax', 'Tmin', 'Wind_speed']
Faltantes: ninguna | valores idénticos a un nuevo pivotado de df_tibaitata_limpio: True

=== 4. Columnas intermedias y ET0 ===
Columnas en el DataFrame: 33 | esperadas: 33
Faltantes: ninguna | no previstas: ninguna | nombres repetidos: ninguno | orden esperado: True

=== 5. Coherencia de unidades ===


,unidad,mín. observado,máx. observado,rango plausible,fuera de rango
variable,,,,,
RH_mean,%,12.929,97.295,"[0, 100]",0
Radiation_Wh_m2,Wh/m²/día,233.000,7461.600,"[0, 1.2e+04]",0
Tmax,°C,14.400,25.200,"[-10, 45]",0
Tmin,°C,-3.600,13.200,"[-15, 35]",0
Wind_speed,m/s,0.763,19.919,"[0, 20]",0
T_mean,°C,9.100,17.400,"[-10, 40]",0
es_tmax,kPa,1.641,3.206,"[0, 10]",0
es_tmin,kPa,0.468,1.517,"[0, 10]",0
es,kPa,1.283,2.190,"[0, 10]",0


  [OK] Rs = Radiation_Wh_m2 * 0.0036 (Wh -> MJ)
  [OK] T_mean = (Tmax + Tmin) / 2
  [OK] Tmax_K = Tmax + 273.16 y Tmin_K = Tmin + 273.16
  [OK] ea = es * RH_mean / 100 (RH en %)
  [OK] gamma = 0.000665 * P
  [OK] J = día del año de Fecha
  [OK] Rns = 0.77 * Rs
  [OK] Rn = Rns - Rnl
  [OK] Rs_Rso = min(Rs_Rso_sin_limitar, 1)
  Informativo: ea > es: 0 | Tmin > Tmax: 0 | Rs > Ra: 0

=== 6. Valores infinitos ===


... (3 outputs left out)

Total de nulos en el DataFrame: 0 | columnas con nulos: 0
Filas con las cinco variables originales ausentes (se conservan): 0

=== 8. Registros conservados ===


Filas en et0: 2909 | fechas únicas en limpio: 2909 | fechas únicas en et0: 2909
df_tibaitata_limpio: (14545, 9) en memoria vs (14545, 9) en disco | idéntico: True

=== 9. Wind_speed usada directamente como u2 ===
ET0 almacenada = ET0 recalculada con u2 = Wind_speed: True
Contraprueba: si se hubiera convertido el viento de 10 m a 2 m, la diferencia máxima sería 0.462 mm/día
Columnas de viento adicionales (u2, u10, ...): ninguna

=== 10. ET0 en mm/día ===
ET0: mín = 0.907 | mediana = 2.619 | media = 2.652 | máx = 4.449 mm/día
Mediana de ET0 / (0.408 * Rn): 0.68 (0.408 convierte MJ/m²/día a mm/día)

=== 11. ET0 negativa (solo se reporta) ===
ET0 < 0: 0 | negativas en el recálculo independiente: 0 | ET0 == 0: 0
No hay registros con ET0 negativa.

=== 12. La auditoría no modificó los datos ===

=== Resultado de la auditoría ===


,verificación,estado,detalle
n,,,
0,"Datos de la estación tomados de df_estacion (altitud, latitud)",OK,"z = 2543 m, latitud = 4.688662°; P, gamma, phi_rad y Rso/Ra coinciden"
1,Todas las fechas originales están representadas,OK,2909 fechas en limpio; 0 ausentes y 0 sobrantes en et0
2,No existen fechas duplicadas,OK,0 fechas duplicadas
2b,Fechas en orden cronológico creciente,OK,orden creciente
3,Las cinco variables meteorológicas originales siguen presentes y sin alterar,OK,faltantes: ninguna; valores idénticos al pivotado independiente: True
4,Todas las columnas intermedias y ET0 están presentes,OK,faltantes: ninguna; repetidas: ninguna
4b,No hay columnas no previstas y el orden es el esperado,OK,no previstas: ninguna; orden esperado: True
5,Valores de cada variable dentro de rangos plausibles para su unidad,OK,0 valores fuera de rango en 0 variables
5b,Relaciones entre variables y conversiones de unidades correctas,OK,todas las identidades se cumplen


Sin errores estructurales (0 advertencias). Se puede exportar.


## Exportación y verificación

Si la auditoría no tiene errores estructurales, se guarda `df_tibaitata_et0` en `./Datasets/Tibaitata_ET0_FAO56.pkl`, se vuelve a cargar y se compara con el DataFrame en memoria (dimensiones, columnas, tipos y valores). `df_tibaitata_limpio` no se modifica.

In [22]:
import os

EXPORT_PATH = "./Datasets/Tibaitata_ET0_FAO56.pkl"

if not AUDITORIA_APTA_PARA_EXPORTAR:
    raise RuntimeError(
        "La auditoría tiene errores estructurales; no se exporta:\n" + "\n".join(errores_estructurales)
    )

# Se exporta solo df_tibaitata_et0; df_tibaitata_limpio no se toca.
df_tibaitata_et0.to_pickle(EXPORT_PATH)
df_et0_recargado = pd.read_pickle(EXPORT_PATH)

verificaciones_export = {
    "El archivo existe y no está vacío": os.path.isfile(EXPORT_PATH) and os.path.getsize(EXPORT_PATH) > 0,
    "Dimensiones iguales": df_et0_recargado.shape == df_tibaitata_et0.shape,
    "Columnas iguales y en el mismo orden": df_et0_recargado.columns.tolist() == df_tibaitata_et0.columns.tolist(),
    "Tipos de datos iguales": df_et0_recargado.dtypes.equals(df_tibaitata_et0.dtypes),
    "Índice igual": df_et0_recargado.index.equals(df_tibaitata_et0.index),
    "Nulos por columna iguales": df_et0_recargado.isna().sum().equals(df_tibaitata_et0.isna().sum()),
}
try:
    pd.testing.assert_frame_equal(df_et0_recargado, df_tibaitata_et0, check_exact=True)
    verificaciones_export["Valores idénticos (comparación exacta)"] = True
except AssertionError as error:
    verificaciones_export["Valores idénticos (comparación exacta)"] = False
    print("Diferencia detectada:", error)
verificaciones_export["df_tibaitata_limpio sigue igual al original"] = (
    df_tibaitata_limpio.equals(pd.read_pickle(DATASET_LIMPIO_PATH))
)

for descripcion, ok in verificaciones_export.items():
    print(f"[{'OK' if ok else 'ERROR'}] {descripcion}")
VERIFICACION_EXPORTACION_OK = all(verificaciones_export.values())
assert VERIFICACION_EXPORTACION_OK, "La verificación posterior a la exportación falló."
print(f"\nArchivo exportado: {EXPORT_PATH} ({os.path.getsize(EXPORT_PATH) / 1024:.1f} KiB)")

[OK] El archivo existe y no está vacío
[OK] Dimensiones iguales
[OK] Columnas iguales y en el mismo orden
[OK] Tipos de datos iguales
[OK] Índice igual
[OK] Nulos por columna iguales
[OK] Valores idénticos (comparación exacta)
[OK] df_tibaitata_limpio sigue igual al original

Archivo exportado: ./Datasets/Tibaitata_ET0_FAO56.pkl (741.5 KiB)


## Resumen final

In [23]:
df_resumen = df_et0_recargado  # se resume el archivo tal como quedó guardado
et0_resumen = df_resumen["ET0"]
n_et0_validas = int(et0_resumen.notna().sum())
n_et0_negativas = int((et0_resumen < 0).sum())
nulos_resumen = df_resumen.isna().sum().rename("nulos").to_frame()
nulos_resumen["% del total"] = (nulos_resumen["nulos"] / len(df_resumen) * 100).round(2)

print("RESUMEN FINAL")
print("=" * 60)
print(f"Dimensiones del dataset : {df_resumen.shape} (filas, columnas)")
print(f"Cantidad de registros   : {len(df_resumen)}")
print(f"Rango de fechas         : {df_resumen['Fecha'].min().date()} a {df_resumen['Fecha'].max().date()}")
print(f"ET0 válidas             : {n_et0_validas}")
print(f"ET0 negativas           : {n_et0_negativas}")
print(f"Estación                : {df_estacion['nombre'].iloc[0]} | z = {alt_est:g} m | latitud = {lat_est}° | longitud = {lon_est}°")
print(f"Ruta de exportación     : {EXPORT_PATH}")
print(f"Verificación posterior  : {'CORRECTA (dimensiones, columnas, tipos y valores conservados)' if VERIFICACION_EXPORTACION_OK else 'FALLÓ'}")
print(f"\nColumnas ({len(df_resumen.columns)}):")
print(df_resumen.columns.tolist())
print("\nValores nulos por columna:")
with pd.option_context("display.max_rows", None):
    display(nulos_resumen)

RESUMEN FINAL
Dimensiones del dataset : (2909, 33) (filas, columnas)
Cantidad de registros   : 2909
Rango de fechas         : 2005-02-02 a 2022-10-31
ET0 válidas             : 2909
ET0 negativas           : 0
Estación                : TIBAITATA [21205420] | z = 2543 m | latitud = 4.688662° | longitud = -74.205626°
Ruta de exportación     : ./Datasets/Tibaitata_ET0_FAO56.pkl
Verificación posterior  : CORRECTA (dimensiones, columnas, tipos y valores conservados)

Columnas (33):
['Fecha', 'RH_mean', 'Radiation_Wh_m2', 'Tmax', 'Tmin', 'Wind_speed', 'T_mean', 'es_tmax', 'es_tmin', 'es', 'es_tmean', 'delta', 'ea', 'P', 'gamma', 'Rs', 'J', 'phi_rad', 'dr', 'solar_declination', 'omega_s', 'Ra', 'Rso', 'Rns', 'Tmax_K', 'Tmin_K', 'ea_sqrt', 'Rs_Rso_sin_limitar', 'Rs_Rso', 'Rnl', 'Rn', 'G', 'ET0']

Valores nulos por columna:


,nulos,% del total
Fecha,0,0.0
RH_mean,0,0.0
Radiation_Wh_m2,0,0.0
Tmax,0,0.0
Tmin,0,0.0
Wind_speed,0,0.0
T_mean,0,0.0
es_tmax,0,0.0
es_tmin,0,0.0
es,0,0.0
